# Segmentação de tecidos em úlceras do pé diabético (DFUTissue) — pipeline alternativo

**Objetivo científico.** Segmentar *Fibrin*, *Granulation* e *Callus* (mais fundo) no dataset **DFUTissue**
(110 imagens anotadas: 78 treino / 16 validação / 16 teste; 600 imagens sem anotação) e comparar, **sem fuga de
informação**, com o artigo *Wound Tissue Segmentation in Diabetic Foot Ulcer Images Using Deep Learning: A Pilot Study*
(Dhar et al., [arXiv:2406.16012](https://arxiv.org/abs/2406.16012); código/dados em
[uwm-bigdata/DFUTissueSegNet](https://github.com/uwm-bigdata/DFUTissueSegNet)).

**Estado desta entrega (leia primeiro).**
* O código foi escrito e **testado de ponta a ponta em CPU, em modo `smoke`, com os dados reais**, fora do Kaggle.
  **Não foi executado em GPU Kaggle** e **nenhum resultado de desempenho é fornecido aqui**: todas as tabelas,
  gráficos e conclusões desta página são gerados quando *você* executa o notebook.
* Os resultados do artigo (tabela abaixo) são constantes copiadas das Tabelas 3–5 do artigo, rotuladas como tal.
* O notebook só declara que ultrapassou o artigo se as métricas calculadas na sessão o mostrarem, e mostra sempre o
  intervalo de confiança *bootstrap* sobre as 16 imagens de teste (um teste tão pequeno é muito ruidoso).

## 1. O que o artigo reporta e como as métricas são calculadas

| Métrica (artigo) | SL | SSL |
|---|---|---|
| Dice global | 84,89 | **87,64** |
| IoU / Precision / Recall (SSL) | 73,75 / 87,69 / 82,27 (SL) | 77,99 / 88,14 / 87,14 |
| Dice Fibrin / Granulation / Callus (SSL) | — | 69,01 / 94,11 / 78,27 |

**Achado crítico para a comparação.** A média aritmética dos três Dice por classe do artigo (SSL) é 80,46, não 87,64, e a
*precision* global (88,14) é superior à média das três *precision* por classe (81,75). Logo o "global" **não** é um macro-Dice
dos tecidos. Lendo o código dos autores (`Codes/supervised_training.ipynb`, célula de inferência) verifica-se que:

1. as máscaras são índices `0=fundo, 1=fibrina, 2=granulação, 3=calo` e o fundo é **excluído** das métricas;
2. as contagens `tp/fp/fn` são acumuladas por imagem **sem serem repostas entre classes** e somadas ao total global
   (`stp += tp` dentro do ciclo de classes), o que dá mais peso às classes processadas depois e **não é um micro-Dice padrão**;
3. as métricas por classe usam também esses acumuladores, e classes ausentes em ambos (GT e predição) são ignoradas.

Por isso este notebook calcula, sobre a mesma matriz de confusão e **com definições explícitas**:
*(a)* Dice por classe agregado sobre o dataset; *(b)* macro-Dice dos 3 tecidos; *(c)* micro-Dice dos tecidos;
*(d)* macro com fundo; *(e)* a média por imagem; e *(f)* uma **réplica fiel do protocolo do código dos autores**
(validada contra uma porta literal do ciclo original). A *(f)* é a única comparável ao 87,64 do artigo, e mesmo assim
a equivalência é uma **hipótese** (não pude confirmar que a tabela do artigo foi gerada exatamente com esse código).
A métrica de seleção de modelos usa *(b)*, que é a mais exigente para a fibrina.

## 2. Análise dos problemas e estratégia escolhida

Problemas identificados (artigo + auditoria dos dados reais): poucos dados anotados (78 treino); fibrina = 0,9 % dos
píxeis (≈12 % do primeiro plano); fronteiras ambíguas fibrina/granulação e calo/pele; variação de cor e iluminação;
e — descoberto na auditoria — **as imagens "Padded" são as originais (67–266 px) coladas no canto de uma tela preta 256×256**,
pelo que numa imagem pequena a ferida ocupa uma fração minúscula do que a rede vê e a resolução efetiva é baixa.

| Decisão | Justificação | Custo / risco |
|---|---|---|
| **ROI + *letterbox***: recortar o conteúdo, ampliá-lo para preencher a tela (256 ou 384) e reverter na avaliação | aumenta a resolução efetiva sem inventar detalhe; mais *tokens* úteis para encoders com *stride* 32; avaliado também no frame original | ampliar não recupera detalhe; testado contra o modo `padded` (réplica do artigo) na validação |
| **Modelos complementares**: U-Net++ + EfficientNet-B4 (CNN), UPerNet + ConvNeXt-S IN-22k (CNN moderna, contexto multiescala), SegFormer MiT-B3 (Transformer) | diversidade de vieses indutivos para *ensemble*; todos com pesos pré-treinados | MiT-B3 é o encoder do artigo, mas aqui com decoder diferente (não o P-scSE) e como membro de ensemble |
| **Transfer learning**: *freeze* inicial do encoder, *fine-tuning* com LR diferencial (encoder ≪ decoder), AdamW, *warmup*+cosseno, EMA | poucos dados → regularizar a adaptação | LR/épocas podem sobreajustar a validação de 16 imagens (mitigado por sementes múltiplas) |
| **Augmentation clinicamente plausível**, implementada em OpenCV/NumPy (máscaras sempre com *nearest*): flips, afim moderada, fotométrica suave, ruído/desfoque; sem *elastic* nem *copy-paste* | evitar tecidos anatomicamente implausíveis; sem dependência de versão do Albumentations | menos diversidade que métodos agressivos |
| **Perda**: CE (pesos raiz-inversa limitados) + Dice nos tecidos; alternativas (Focal+Dice, Focal-Tversky, +Lovász) comparadas na validação; *class-aware sampling* para fibrina | desequilíbrio sem sobreponderar (falsos positivos) | — |
| **SSL**: *EMA teacher–student* online, vistas fraca→forte (fotométrica + CutMix), limiares de confiança **por classe** adaptativos, ignorar píxeis incertos, ramp-up do peso, rondas iterativas com *rollback* se a validação piorar, exclusão de quase-duplicados de val/teste | substitui a escolha aleatória de *pseudo-labels* do artigo; proteção contra *confirmation bias* | só ajuda se o modelo supervisionado já for razoável; validado, não assumido |
| **Inferência**: TTA (flips ± multiescala), *ensemble* por probabilidades, remoção de componentes pequenos — tudo escolhido **só na validação** | ganhos pequenos e baratos | validação pequena → ganho estimado otimista |

**Não implementado deliberadamente** (para não apresentar o que não foi feito): *self-supervised pre-training* (600 imagens de 256 px não
justificam o custo face a ImageNet/IN-22k), DINOv2 (depende de pesos externos/Internet), Optuna (validação de 16 imagens
sobreajusta com HPO) e a tarefa binária *Chronic Wound* (experiência distinta, não misturável com a multiclasse).

**Disciplina anti-fuga.** O teste só é lido na fase final, através de um *ledger* (`TEST_LEDGER`) que regista cada
avaliação; não participa em seleção, *pseudo-labels*, limiares nem pós-processamento. Os 600 não anotados são filtrados
por *perceptual hash* contra validação e teste.

**Como usar no Kaggle.** (1) Crie um *Dataset* Kaggle com a pasta `DFUTissue` do repositório (contém `Labeled/` e
`Unlabeled/`) e adicione-o ao notebook; (2) ative a GPU; (3) escolha `MODE` na célula de configuração
(`quick` ≈ primeira corrida curta; `full` = experiência completa); (4) com Internet desligada, adicione também um dataset com os
*wheels* de `segmentation_models_pytorch` e os pesos pré-treinados (ver célula de ambiente e de pesos).

## 2–4. Ambiente Kaggle, dependências e hardware

Esta célula funciona com Internet ligada (instala o que faltar) ou desligada (procura *wheels* em `/kaggle/input`).
Nunca usa credenciais. Em Kaggle os resultados vão para `/kaggle/working`; fora do Kaggle usam-se as variáveis
`DFU_INPUT` (dados) e `DFU_WORK` (saídas) ou pastas locais.

In [ ]:
import os, sys, platform, subprocess, importlib, json, time, math, random, shutil, hashlib, glob, warnings, itertools, copy, socket
from pathlib import Path
from collections import OrderedDict, defaultdict

warnings.filterwarnings("ignore", category=UserWarning)

IS_KAGGLE = Path("/kaggle/input").exists()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path(os.environ.get("DFU_INPUT", "."))
WORK = Path("/kaggle/working") if IS_KAGGLE else Path(os.environ.get("DFU_WORK", "./working"))
WORK.mkdir(parents=True, exist_ok=True)


def has_internet(host: str = "pypi.org", port: int = 443, timeout: float = 4.0) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


INTERNET = has_internet()


def _version_ok(have: str, need: str | None) -> bool:
    if need is None:
        return True
    from packaging.version import Version
    return Version(have.split("+")[0]) >= Version(need)


def find_wheels(stem: str, max_depth: int = 4) -> list[Path]:
    """Procura *wheels* locais (Kaggle Datasets) sem varrer ficheiros de dados."""
    out: list[Path] = []
    if not INPUT_ROOT.exists():
        return out
    base_depth = len(INPUT_ROOT.parts)
    for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
        if len(Path(dirpath).parts) - base_depth >= max_depth:
            dirnames[:] = []
        out += [Path(dirpath) / f for f in filenames if f.endswith(".whl") and f.lower().startswith(stem.lower())]
    return out


def ensure_package(import_name: str, pip_name: str | None = None, min_version: str | None = None) -> None:
    pip_name = pip_name or import_name
    try:
        mod = importlib.import_module(import_name)
        if _version_ok(getattr(mod, "__version__", "0"), min_version):
            return
    except ImportError:
        pass
    spec = f"{pip_name}>={min_version}" if min_version else pip_name
    if INTERNET:
        cmd = [sys.executable, "-m", "pip", "install", "-q", spec]
    else:
        wheels = find_wheels(pip_name.replace("-", "_"))
        if not wheels:
            raise RuntimeError(
                f"Falta o pacote '{pip_name}' e não há Internet nem wheel em /kaggle/input. "
                f"Ative a Internet no notebook ou adicione um Dataset com o wheel de {pip_name}."
            )
        cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", str(wheels[0])]
    print("A instalar:", " ".join(cmd[3:]))
    subprocess.run(cmd, check=True)
    importlib.invalidate_caches()
    importlib.import_module(import_name)


ensure_package("packaging")
ensure_package("segmentation_models_pytorch", "segmentation-models-pytorch", "0.5.0")  # UPerNet/Segformer exigem >= 0.5
ensure_package("timm", "timm", "0.9.0")
ensure_package("cv2", "opencv-python-headless")
for _imp, _pip in [("pandas", "pandas"), ("matplotlib", "matplotlib"), ("scipy", "scipy"), ("tqdm", "tqdm")]:
    ensure_package(_imp, _pip)

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import segmentation_models_pytorch as smp
import matplotlib

if not IS_KAGGLE and os.environ.get("DFU_HEADLESS", "0") == "1":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

try:
    from IPython.display import display
except ImportError:  # execução fora de Jupyter
    display = print

cv2.setNumThreads(0)  # evita sobre-subscrição com os workers do DataLoader

### Hardware e otimização de recursos
Deteta CUDA, nome/VRAM da GPU e decide *mixed precision*. **T4/P100 não suportam bfloat16 eficiente**, por isso usa-se fp16 com
`GradScaler`. O *batch size* efetivo é fixo (`CFG.batch_size`); o *micro-batch* real é descoberto por sondagem de OOM e compensado
por acumulação de gradientes.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HW = {
    "device": str(DEVICE),
    "cuda_count": torch.cuda.device_count() if DEVICE.type == "cuda" else 0,
    "gpu_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if DEVICE.type == "cuda" else None,
    "cpu_count": os.cpu_count(),
    "python": platform.python_version(),
    "torch": torch.__version__, "timm": timm.__version__, "smp": smp.__version__,
    "opencv": cv2.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "internet": INTERNET, "is_kaggle": IS_KAGGLE,
}
USE_AMP = DEVICE.type == "cuda"
print(json.dumps(HW, indent=2))
if DEVICE.type == "cpu":
    print("AVISO: sem GPU. Em Kaggle ative Settings > Accelerator > GPU T4/P100. O modo 'full' é impraticável em CPU.")

## 5. Configuração global

Escolha **`MODE`**:
* `smoke` — diagnóstico de pipeline (minutos, mesmo em CPU; modelo pequeno com pesos aleatórios; **os números não têm valor científico**);
* `quick` — primeira corrida real em GPU (2 arquiteturas, 1 semente, sem sondas nem ablação);
* `full` — experiência completa (3 arquiteturas × 3 sementes, sondas de *frame*/resolução e de perda, ablação, SSL em 2 rondas, *bootstrap* de 2000).

Todos os hiperparâmetros vivem em `CFG`; o dicionário completo, as sementes e as versões são gravados em `config.json`.

In [ ]:
MODE = os.environ.get("DFU_MODE", "quick")  # 'smoke' | 'quick' | 'full'

_BASE = dict(
    # dados
    frame="roi",            # 'roi' (recorta + letterbox do conteúdo) | 'padded' (tela 256x256 do artigo)
    size=256,               # lado da tela de entrada do modelo
    num_workers=min(4, os.cpu_count() or 1),
    # treino
    batch_size=16, epochs=150, patience=40, warmup_epochs=5, freeze_epochs=3,
    lr_decoder=1e-3, lr_encoder=2e-4, weight_decay=1e-2, ema_decay=0.98, grad_clip=1.0, min_lr_ratio=0.01,
    loss="ce_dice", label_smoothing=0.0, fibrin_boost=0.5, class_weight_cap=5.0,
    # fases opcionais
    do_sanity=True, do_lr_range=True, do_probes=False, do_ablation=False,
    probe_epochs=40, ablation_epochs=40, probe_frames=[("padded", 256), ("roi", 256), ("roi", 384)],
    probe_losses=["ce_dice", "focal_dice", "focal_tversky", "ce_dice_lovasz"],
    # SSL
    do_ssl=True, ssl_rounds=2, ssl_epochs=40, ssl_steps_per_epoch=20, ssl_lambda_u=1.0, ssl_keep_frac=0.7,
    ssl_tau_min=0.75, ssl_tau_max=0.97, ssl_all_seeds=False, ssl_dup_hamming=8,
    # inferência / estatística
    tta_candidates=["none", "flip", "flip_ms"], pp_min_area_fracs=[0.0, 0.0005, 0.001, 0.002, 0.005],
    bootstrap_b=2000, time_budget_hours=9.0, resume=True, seed_base=42,
)
_MODES = {
    "smoke": dict(archs=["smoke_fpn_r18"], seeds=[0], epochs=3, patience=3, warmup_epochs=1, freeze_epochs=1, batch_size=8,
                  ssl_rounds=1, ssl_epochs=2, ssl_steps_per_epoch=3, bootstrap_b=100, do_lr_range=True, num_workers=0,
                  do_probes=True, probe_epochs=1, probe_frames=[("padded", 256), ("roi", 256)],
                  probe_losses=["ce_dice", "focal_tversky"], do_ablation=True, ablation_epochs=1, time_budget_hours=2.0),
    "quick": dict(archs=["unetpp_effb4", "segformer_mitb3"], seeds=[42], epochs=60, patience=20, ssl_rounds=1, ssl_epochs=30,
                  bootstrap_b=1000, time_budget_hours=4.0),
    "full": dict(archs=["unetpp_effb4", "upernet_convnext_s", "segformer_mitb3"], seeds=[42, 43, 44], do_probes=True,
                 do_ablation=True, ssl_all_seeds=False),
}
assert MODE in _MODES, f"MODE inválido: {MODE}"
CFG = {**_BASE, **_MODES[MODE], "mode": MODE}
CFG.update(json.loads(os.environ.get("DFU_CFG_OVERRIDE", "{}")))  # sobreposição opcional por variável de ambiente (JSON)

OUT = WORK / f"dfu_run_{MODE}"
DIRS = {k: OUT / k for k in ["runs", "figures", "tables", "models", "cache", "predictions"]}
for _d in DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Background", "Fibrin", "Granulation", "Callus"]
K = 4
TISSUES = [1, 2, 3]
PALETTE = np.array([[0, 0, 0], [230, 30, 30], [40, 200, 60], [40, 90, 240]], dtype=np.uint8)  # bg, fibrina, granulação, calo

# Resultados reportados no artigo (Tabelas 3, 4 e 5) — constantes copiadas, NÃO resultados desta sessão.
PAPER = {
    "supervised": {"IoU": 73.75, "Precision": 87.69, "Recall": 82.27, "Dice": 84.89},
    "semi_supervised": {"IoU": 77.99, "Precision": 88.14, "Recall": 87.14, "Dice": 87.64},
    "per_class_ssl": {
        "Fibrin": {"IoU": 52.68, "Precision": 71.48, "Recall": 66.70, "Dice": 69.01},
        "Granulation": {"IoU": 88.87, "Precision": 93.12, "Recall": 95.11, "Dice": 94.11},
        "Callus": {"IoU": 64.29, "Precision": 80.65, "Recall": 76.02, "Dice": 78.27},
    },
}


class _Budget:
    """Orçamento de tempo de parede (sessões Kaggle têm limite de duração)."""

    def __init__(self, hours: float):
        self.t0, self.limit = time.time(), hours * 3600

    def left(self) -> float:
        return self.limit - (time.time() - self.t0)

    def ok(self, need_s: float = 0.0) -> bool:
        return self.left() > need_s


BUDGET = _Budget(CFG["time_budget_hours"])


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(CFG["seed_base"])
print(f"MODE={MODE} | saídas em {OUT}")
print(json.dumps({k: v for k, v in CFG.items() if k in ["archs", "seeds", "frame", "size", "epochs", "batch_size", "loss", "do_probes", "do_ssl"]}))

## 6. Localização e auditoria do dataset

O notebook procura em `/kaggle/input` (ou em `DFU_DATA_ROOT`) uma pasta que contenha `Labeled/labeled_train_names.txt`
(estrutura do repositório oficial: `Labeled/{Original,Padded}/{Images,Annotations,Palette}/{TrainVal,Test}`, `Labeled/*_names.txt`
e `Unlabeled/`). **Não inventa dados**: se não os encontrar, para com instruções.

Verificações: contagens, correspondência imagem↔máscara, ficheiros corrompidos, valores únicos e codificação das máscaras
(indexadas `P`, cinzentos `L` ou RGB — **não se assume** um formato), tamanhos, equivalência *Padded* ≙ *Original*,
distribuição de píxeis, duplicados exatos e quase-duplicados (*dHash*) dentro e **entre** divisões, e entre os 600 não anotados e val/teste.

In [ ]:
ALLOW_GIT_DOWNLOAD = os.environ.get("DFU_ALLOW_DOWNLOAD", "0") == "1"  # opt-in; só com Internet ligada
DATA_HELP = """
Dataset DFUTissue não encontrado. Passos para o adicionar ao Kaggle:
  1. Obtenha a pasta 'DFUTissue' de https://github.com/uwm-bigdata/DFUTissueSegNet (contém Labeled/ e Unlabeled/).
  2. Kaggle > Datasets > New Dataset > carregue a pasta (mantenha a estrutura Labeled/ e Unlabeled/).
  3. No notebook: Add Input > o seu dataset. Em alternativa defina DFU_DATA_ROOT ou DFU_ALLOW_DOWNLOAD=1 (requer Internet).
"""


def find_dataset_root() -> Path | None:
    explicit = os.environ.get("DFU_DATA_ROOT")
    bases = [Path(explicit)] if explicit else [INPUT_ROOT]
    for base in bases:
        if not base.exists():
            continue
        depth0 = len(base.parts)
        for dirpath, dirnames, filenames in os.walk(base):
            if len(Path(dirpath).parts) - depth0 > 6:
                dirnames[:] = []
                continue
            if "labeled_train_names.txt" in filenames and Path(dirpath).name == "Labeled":
                return Path(dirpath).parent
    return None


DATA_ROOT = find_dataset_root()
if DATA_ROOT is None and ALLOW_GIT_DOWNLOAD and INTERNET:
    dst = WORK / "DFUTissueSegNet"
    if not dst.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/uwm-bigdata/DFUTissueSegNet.git", str(dst)], check=True)
    DATA_ROOT = dst / "DFUTissue"
if DATA_ROOT is None:
    raise RuntimeError(DATA_HELP)
LAB_DIR = DATA_ROOT / "Labeled"
UNL_DIR = DATA_ROOT / "Unlabeled" if (DATA_ROOT / "Unlabeled").exists() else None
print("DATA_ROOT:", DATA_ROOT, "| Unlabeled:", "sim" if UNL_DIR else "NÃO (SSL será omitido)")


def read_names(fname: str) -> list[str]:
    return [l.strip() for l in (LAB_DIR / fname).read_text().splitlines() if l.strip()]


SPLIT_NAMES = {"train": read_names("labeled_train_names.txt"), "val": read_names("labeled_val_names.txt"), "test": read_names("test_names.txt")}
print({k: len(v) for k, v in SPLIT_NAMES.items()}, "(artigo: 78 / 16 / 16)")


def _locate(frame_dir: str, kind: str, name: str) -> Path | None:
    for sub in ("TrainVal", "Test", "Train", "Val", ""):
        p = LAB_DIR / frame_dir / kind / sub / f"{name}.png"
        if p.exists():
            return p
    return None


MASK_COLORS = {(0, 0, 0): 0, (255, 0, 0): 1, (0, 255, 0): 2, (0, 0, 255): 3}


def read_mask(path: Path) -> tuple[np.ndarray, str]:
    """Lê uma máscara e devolve (índices uint8 [0..3], descrição da codificação). Falha se houver valores inválidos."""
    im = Image.open(path)
    mode, arr = im.mode, np.array(im)
    if mode in ("P", "L", "I", "I;16"):
        idx, enc = arr.astype(np.int64), f"{mode}:indices"
    elif mode in ("RGB", "RGBA"):
        rgb = arr[..., :3]
        if (rgb[..., 0] == rgb[..., 1]).all() and (rgb[..., 1] == rgb[..., 2]).all():
            idx, enc = rgb[..., 0].astype(np.int64), f"{mode}:canais-iguais(indices)"
        else:
            idx = np.full(rgb.shape[:2], -1, np.int64)
            for col, k in MASK_COLORS.items():
                idx[(rgb == np.array(col)).all(-1)] = k
            enc = f"{mode}:cores"
    else:
        raise ValueError(f"Modo de máscara não suportado: {mode} ({path})")
    if idx.min() < 0 or idx.max() > 3:
        raise ValueError(f"Máscara com valores fora de 0..3 ou cores desconhecidas: {path} -> {np.unique(idx)[:10]}")
    return idx.astype(np.uint8), enc


def read_rgb(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"))


def content_extent(img: np.ndarray) -> tuple[int, int]:
    """Altura/largura do conteúdo numa imagem com *zero padding* no canto inferior-direito."""
    nz = img.max(-1) > 0
    rows, cols = np.where(nz.any(1))[0], np.where(nz.any(0))[0]
    return int(rows.max()) + 1, int(cols.max()) + 1


def dhash(img: np.ndarray) -> int:
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    g = cv2.resize(g, (9, 8), interpolation=cv2.INTER_AREA)
    bits = (g[:, 1:] > g[:, :-1]).ravel()
    return int("".join("1" if b else "0" for b in bits), 2)


def hamming(a: int, b: int) -> int:
    return (a ^ b).bit_count()

In [ ]:
audit_rows, problems = [], []
labeled_hash: dict[str, int] = {}
for split, names in SPLIT_NAMES.items():
    for n in names:
        row = dict(name=n, split=split, corrupt=False)
        try:
            pi, pa, pp = _locate("Original", "Images", n), _locate("Original", "Annotations", n), _locate("Original", "Palette", n)
            qi, qa, qp = _locate("Padded", "Images", n), _locate("Padded", "Annotations", n), _locate("Padded", "Palette", n)
            if pi is None or pa is None or qi is None or (qa is None and qp is None):
                raise FileNotFoundError("imagem/máscara em falta")
            img = read_rgb(pi)
            ann, enc_a = read_mask(pa)
            row.update(h=img.shape[0], w=img.shape[1], enc_orig_annotation=enc_a, size_mismatch=img.shape[:2] != ann.shape)
            if pp is not None:
                pal, enc_p = read_mask(pp)
                row.update(enc_orig_palette=enc_p, annot_eq_palette=bool((pal == ann).all()))
            pimg = read_rgb(qi)
            pm, enc_pm = read_mask(qa or qp)
            row.update(enc_padded_mask=enc_pm, padded_shape=pimg.shape[:2])
            h, w = img.shape[:2]
            if max(h, w) <= pimg.shape[0]:
                row["padded_img_eq_orig"] = bool((pimg[:h, :w] == img).all() and (pimg[h:] == 0).all() and (pimg[:, w:] == 0).all())
                row["padded_mask_eq_orig"] = bool((pm[:h, :w] == ann).all() and (pm[h:] == 0).all() and (pm[:, w:] == 0).all())
                row["extent_detected_ok"] = content_extent(pimg) == (h, w)
            else:  # imagens > 256 foram reduzidas no 'Padded' (documentado no relatório)
                row["padded_img_eq_orig"] = row["padded_mask_eq_orig"] = row["extent_detected_ok"] = None
            counts = np.bincount(ann.ravel(), minlength=4)[:4]
            row.update({f"px_{c.lower()}": int(v) for c, v in zip(CLASS_NAMES, counts)})
            labeled_hash[n] = dhash(img)
        except Exception as e:  # noqa: BLE001 — a auditoria deve reportar, não rebentar
            row.update(corrupt=True, error=f"{type(e).__name__}: {e}")
            problems.append((n, row["error"]))
        audit_rows.append(row)

AUDIT = pd.DataFrame(audit_rows)
AUDIT.to_csv(DIRS["tables"] / "audit_labeled.csv", index=False)
print(f"Imagens anotadas: {len(AUDIT)} | corrompidas/inválidas: {int(AUDIT.corrupt.sum())}")
for n, e in problems:
    print("  PROBLEMA", n, e)
assert not AUDIT.corrupt.any(), "Há ficheiros inválidos — corrija o dataset antes de treinar."
print("Codificações (Original/Annotations):", AUDIT.enc_orig_annotation.value_counts().to_dict())
print("Codificações (Padded mask):", AUDIT.enc_padded_mask.value_counts().to_dict())
if "annot_eq_palette" in AUDIT:
    print("Annotations == Palette em todas:", bool(AUDIT.annot_eq_palette.dropna().all()))
print("Imagem/máscara com tamanhos diferentes:", int(AUDIT.size_mismatch.sum()))
print("Padded == Original (conteúdo no canto sup-esq.):", AUDIT.padded_img_eq_orig.dropna().all(), "| máscaras:", AUDIT.padded_mask_eq_orig.dropna().all(),
      "| deteção de extensão correta:", AUDIT.extent_detected_ok.dropna().all(), f"| imagens >256 (não comparáveis): {int(AUDIT.padded_img_eq_orig.isna().sum())}")
print("Tamanhos originais: min", AUDIT[["h", "w"]].min().tolist(), "max", AUDIT[["h", "w"]].max().tolist(), "| mediana", AUDIT[["h", "w"]].median().tolist())

# divisões: sobreposição por nome
sets = {k: set(v) for k, v in SPLIT_NAMES.items()}
overlaps = {f"{a}&{b}": sorted(sets[a] & sets[b]) for a, b in [("train", "val"), ("train", "test"), ("val", "test")]}
print("Sobreposição de nomes entre divisões:", overlaps)
assert all(len(v) == 0 for v in overlaps.values()), "Fuga: nomes repetidos entre divisões"

In [ ]:
# Distribuição de píxeis por classe e por divisão (frame original)
px_cols = [f"px_{c.lower()}" for c in CLASS_NAMES]
share = AUDIT.groupby("split")[px_cols].sum()
share_pct = share.div(share.sum(axis=1), axis=0) * 100
presence = AUDIT.groupby("split")[px_cols].agg(lambda s: int((s > 0).sum()))
fg_share = share[px_cols[1:]].div(share[px_cols[1:]].sum(axis=1), axis=0) * 100
print("% de píxeis por classe (inclui fundo):")
display(share_pct.round(2))
print("% de píxeis do primeiro plano:")
display(fg_share.round(1))
print("Nº de imagens com cada classe:")
display(presence)

# Duplicados exatos e quase-duplicados (dHash 64 bits). Limiar: distância de Hamming <= 6 -> a rever manualmente.
NEAR_DUP_T = 6
names_all = list(labeled_hash)
split_of = AUDIT.set_index("name").split.to_dict()
dups = []
for a, b in itertools.combinations(names_all, 2):
    d = hamming(labeled_hash[a], labeled_hash[b])
    if d <= NEAR_DUP_T:
        dups.append(dict(a=a, b=b, split_a=split_of[a], split_b=split_of[b], hamming=d))
DUPS = pd.DataFrame(dups, columns=["a", "b", "split_a", "split_b", "hamming"])
print(f"Pares de imagens anotadas com dHash <= {NEAR_DUP_T}: {len(DUPS)} | entre divisões diferentes: {int((DUPS.split_a != DUPS.split_b).sum()) if len(DUPS) else 0}")
if len(DUPS):
    display(DUPS)
print("NOTA: sem identificador de paciente não é possível garantir separação por paciente (várias imagens podem vir do mesmo doente);"
      " isto é uma limitação, partilhada com o artigo, que afeta a generalização estimada.")

In [ ]:
# Imagens sem anotação: leitura robusta, extensão do conteúdo e exclusão de quase-duplicados de val/teste (anti-fuga no SSL)
UNL = []
EXCLUDED_UNL = []
if UNL_DIR is not None:
    heldout_hash = {n: labeled_hash[n] for n in SPLIT_NAMES["val"] + SPLIT_NAMES["test"] if n in labeled_hash}
    for p in sorted(UNL_DIR.glob("*.png")):
        try:
            img = read_rgb(p)
            h, w = content_extent(img)
            hsh = dhash(img[:h, :w])
            dmin, nearest = min((hamming(hsh, hv), hn) for hn, hv in heldout_hash.items())
            rec = dict(name=p.stem, path=p, h=h, w=w, dmin_heldout=dmin, nearest=nearest)
            (EXCLUDED_UNL if dmin <= CFG["ssl_dup_hamming"] else UNL).append(rec)
        except Exception as e:  # noqa: BLE001
            print("Não anotada ilegível:", p.name, e)
    print(f"Não anotadas: {len(UNL) + len(EXCLUDED_UNL)} | excluídas por quase-duplicado de val/teste (Hamming <= {CFG['ssl_dup_hamming']}): {len(EXCLUDED_UNL)} | usáveis: {len(UNL)}")
    pd.DataFrame(UNL + EXCLUDED_UNL).drop(columns=["path"]).to_csv(DIRS["tables"] / "audit_unlabeled.csv", index=False)
    if EXCLUDED_UNL:
        display(pd.DataFrame(EXCLUDED_UNL).drop(columns=["path"]).head(10))
else:
    print("Sem pasta Unlabeled: o SSL será omitido e o notebook segue em modo supervisionado.")

## 7. Análise exploratória
Imagens originais, máscaras, *overlays* e distribuição de classes. Os dois **frames** de avaliação/treino são:
* `padded` — tela 256×256 do artigo (conteúdo no canto, resto preto);
* `roi` — apenas o conteúdo original, depois ampliado (*letterbox*) para a tela do modelo. A avaliação reverte para o tamanho original.

In [ ]:
def load_frame_records(frame: str) -> dict[str, list[dict]]:
    """Carrega imagens+máscaras anotadas num dado frame: {'train': [...], 'val': [...], 'test': [...]}."""
    assert frame in ("roi", "padded")
    folder = "Original" if frame == "roi" else "Padded"
    out: dict[str, list[dict]] = {}
    for split, names in SPLIT_NAMES.items():
        recs = []
        for n in names:
            img = read_rgb(_locate(folder, "Images", n))
            mpath = _locate(folder, "Palette", n) or _locate(folder, "Annotations", n)
            mask, _ = read_mask(mpath)
            assert img.shape[:2] == mask.shape, f"{n}: imagem {img.shape[:2]} != máscara {mask.shape}"
            recs.append(dict(name=n, img=img, mask=mask))
        out[split] = recs
    return out


def load_unlabeled_records(frame: str) -> list[dict]:
    recs = []
    for r in UNL:
        img = read_rgb(r["path"])
        if frame == "roi":
            img = np.ascontiguousarray(img[: r["h"], : r["w"]])
        recs.append(dict(name=r["name"], img=img))
    return recs


_FRAME_CACHE: dict[str, dict] = {}


def get_frame(frame: str) -> dict[str, list[dict]]:
    if frame not in _FRAME_CACHE:
        _FRAME_CACHE[frame] = load_frame_records(frame)
    return _FRAME_CACHE[frame]


def colorize(mask: np.ndarray) -> np.ndarray:
    return PALETTE[mask]


def overlay(img: np.ndarray, mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    out = img.astype(np.float32).copy()
    fg = mask > 0
    out[fg] = (1 - alpha) * out[fg] + alpha * PALETTE[mask][fg]
    return out.astype(np.uint8)


def savefig(fig, name: str) -> None:
    fig.savefig(DIRS["figures"] / name, dpi=130, bbox_inches="tight")


_orig = get_frame("roi")
fig, axes = plt.subplots(3, 6, figsize=(16, 8.5))
for j, rec in enumerate(_orig["train"][:6]):
    axes[0, j].imshow(rec["img"]); axes[0, j].set_title(f"{rec['name']}  {rec['img'].shape[1]}x{rec['img'].shape[0]}", fontsize=9)
    axes[1, j].imshow(colorize(rec["mask"]))
    axes[2, j].imshow(overlay(rec["img"], rec["mask"]))
    for i in range(3):
        axes[i, j].axis("off")
axes[0, 0].set_ylabel("imagem"); fig.suptitle("Treino — imagem / máscara (vermelho=fibrina, verde=granulação, azul=calo) / overlay (frame original)")
fig.tight_layout(); savefig(fig, "eda_samples.png"); plt.show()

In [ ]:
# Distribuição de classes, tamanhos e o efeito do padding
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
share.T.plot.bar(ax=ax[0], logy=True, rot=20, legend=True); ax[0].set_title("Píxeis por classe e divisão (log)")
presence.T.plot.bar(ax=ax[1], rot=20); ax[1].set_title("Nº de imagens com a classe")
ax[2].hist(AUDIT.h, bins=20); ax[2].set_title("Altura original (px)"); ax[2].axvline(256, color="r", ls="--")
_content_frac = (AUDIT.h * AUDIT.w).clip(upper=256 * 256) / (256 * 256)
ax[3].hist(_content_frac, bins=20); ax[3].set_title("Fração da tela 256x256 ocupada pelo conteúdo\n(no frame 'padded')")
fig.tight_layout(); savefig(fig, "eda_distribution.png"); plt.show()
print(f"Mediana da fração útil da tela no frame 'padded': {_content_frac.median():.2f} (o resto é preto)")

if UNL:
    fig, axes = plt.subplots(1, 6, figsize=(16, 3))
    for a, r in zip(axes, UNL[:6]):
        a.imshow(read_rgb(r["path"])[: r["h"], : r["w"]]); a.set_title(f"não anotada {r['name']}", fontsize=8); a.axis("off")
    fig.tight_layout(); savefig(fig, "eda_unlabeled.png"); plt.show()

## 8–9. Pré-processamento, *augmentation*, *Datasets* e *DataLoaders*

* **Letterbox**: o conteúdo (altura×largura originais) é ampliado/reduzido mantendo a proporção até caber numa tela `S×S` e centrado.
  Imagens e máscaras partilham exatamente a mesma geometria; **máscaras usam sempre *nearest***, probabilidades usam *bilinear* só ao reverter.
* **Augmentation** própria (OpenCV/NumPy, sem dependência de versão de bibliotecas): flips, afim moderada (rotação ±25°, escala 0,8–1,25,
  translação ±8 %, cisalhamento ±5°), HSV suave (matiz ±5 em escala 0–179 de OpenCV), brilho/contraste, gamma, ruído e desfoque.
  A fotometria só altera a **área válida** (o preto do *padding* mantém-se preto, como nas imagens de teste). Não se usam *elastic/grid distortion*
  (alteram fronteiras já ambíguas) nem *copy-paste* (criaria tecidos anatomicamente incoerentes). Se uma transformação geométrica destruir
  >60 % do primeiro plano, repete-se (preserva a região de interesse). **Nada aleatório é aplicado na avaliação.**
* **Class-aware sampling**: imagens com mais fibrina têm maior probabilidade de amostragem (`fibrin_boost`).

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def letterbox_geom(h: int, w: int, S: int) -> dict:
    s = S / max(h, w)
    nh, nw = max(1, min(S, round(h * s))), max(1, min(S, round(w * s)))
    return dict(h=h, w=w, S=S, scale=s, nh=nh, nw=nw, y0=(S - nh) // 2, x0=(S - nw) // 2)


def apply_letterbox(img: np.ndarray, mask: np.ndarray | None, g: dict):
    """Devolve (tela imagem uint8 SxSx3, tela máscara uint8 SxS | None, máscara de validade uint8 SxS)."""
    S, nh, nw, y0, x0 = g["S"], g["nh"], g["nw"], g["y0"], g["x0"]
    interp = cv2.INTER_AREA if g["scale"] < 1 else cv2.INTER_CUBIC
    canvas = np.zeros((S, S, 3), np.uint8)
    canvas[y0:y0 + nh, x0:x0 + nw] = np.clip(cv2.resize(img, (nw, nh), interpolation=interp), 0, 255)
    cm = None
    if mask is not None:
        cm = np.zeros((S, S), np.uint8)
        cm[y0:y0 + nh, x0:x0 + nw] = cv2.resize(mask, (nw, nh), interpolation=cv2.INTER_NEAREST)  # NEAREST: rótulos categóricos
    valid = np.zeros((S, S), np.uint8)
    valid[y0:y0 + nh, x0:x0 + nw] = 1
    return canvas, cm, valid


def unletterbox_probs(probs: torch.Tensor, g: dict) -> torch.Tensor:
    """(C,S,S) -> (C,h,w): recorta a área do conteúdo e repõe o tamanho original (bilinear nas probabilidades)."""
    crop = probs[:, g["y0"]:g["y0"] + g["nh"], g["x0"]:g["x0"] + g["nw"]]
    if (g["nh"], g["nw"]) == (g["h"], g["w"]):
        return crop
    return F.interpolate(crop[None].float(), size=(g["h"], g["w"]), mode="bilinear", align_corners=False)[0]


AUG_LEVELS = {
    "none": dict(),
    "flip": dict(flip_h=0.5, flip_v=0.5),
    "weak": dict(flip_h=0.5, flip_v=0.5, affine_p=0.5, rot=10, scale=(0.9, 1.1), trans=0.04, shear=0),
    "standard": dict(flip_h=0.5, flip_v=0.5, affine_p=0.8, rot=25, scale=(0.8, 1.25), trans=0.08, shear=5,
                     hsv_p=0.4, bc_p=0.6, gamma_p=0.3, noise_p=0.25, blur_p=0.2),
}


class Augmenter:
    """Augmentation geométrica+fotométrica; imagem e máscara são alinhadas pela mesma matriz afim."""

    def __init__(self, level: str = "standard"):
        self.p = AUG_LEVELS[level]
        self.level = level

    def __call__(self, img: np.ndarray, mask: np.ndarray | None, valid: np.ndarray, rng: np.random.Generator):
        p, S = self.p, img.shape[0]
        if not p:
            return img, mask, valid
        if rng.random() < p.get("flip_h", 0):
            img, valid = img[:, ::-1], valid[:, ::-1]
            mask = mask[:, ::-1] if mask is not None else None
        if rng.random() < p.get("flip_v", 0):
            img, valid = img[::-1], valid[::-1]
            mask = mask[::-1] if mask is not None else None
        img, valid = np.ascontiguousarray(img), np.ascontiguousarray(valid)
        mask = np.ascontiguousarray(mask) if mask is not None else None
        if rng.random() < p.get("affine_p", 0):
            fg0 = int((mask > 0).sum()) if mask is not None else 0
            for _ in range(4):
                ang = rng.uniform(-p["rot"], p["rot"])
                sc = math.exp(rng.uniform(math.log(p["scale"][0]), math.log(p["scale"][1])))
                sh = math.radians(rng.uniform(-p["shear"], p["shear"])) if p["shear"] else 0.0
                tx, ty = rng.uniform(-p["trans"], p["trans"], 2) * S
                M = cv2.getRotationMatrix2D((S / 2, S / 2), ang, sc)
                M = np.vstack([M, [0, 0, 1]]) @ np.array([[1, math.tan(sh), -math.tan(sh) * S / 2], [0, 1, 0], [0, 0, 1]])
                M[0, 2] += tx
                M[1, 2] += ty
                M = M[:2]
                m2 = cv2.warpAffine(mask, M, (S, S), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0) if mask is not None else None
                if m2 is None or fg0 == 0 or (m2 > 0).sum() >= 0.4 * fg0:
                    break
            img = cv2.warpAffine(img, M, (S, S), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
            valid = cv2.warpAffine(valid, M, (S, S), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            mask = m2
        if "hsv_p" in p or "bc_p" in p:
            img = self._photometric(img, valid, rng)
        return img, mask, valid

    def _photometric(self, img: np.ndarray, valid: np.ndarray, rng: np.random.Generator) -> np.ndarray:
        p = self.p
        if rng.random() < p["hsv_p"]:
            hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[..., 0] = (hsv[..., 0] + rng.uniform(-5, 5)) % 180
            hsv[..., 1] = np.clip(hsv[..., 1] * rng.uniform(0.85, 1.15), 0, 255)
            hsv[..., 2] = np.clip(hsv[..., 2] * rng.uniform(0.9, 1.1), 0, 255)
            img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        x = img.astype(np.float32) / 255.0
        if rng.random() < p["bc_p"]:
            x = (x - 0.5) * rng.uniform(0.8, 1.2) + 0.5 + rng.uniform(-0.1, 0.1)
        if rng.random() < p["gamma_p"]:
            x = np.clip(x, 0, 1) ** rng.uniform(0.8, 1.25)
        if rng.random() < p["blur_p"]:
            x = cv2.GaussianBlur(x, (0, 0), rng.uniform(0.4, 1.0))
        if rng.random() < p["noise_p"]:
            x = x + rng.normal(0, rng.uniform(0.005, 0.03), x.shape).astype(np.float32)
        x = np.clip(x, 0, 1) * valid[..., None]  # o padding mantém-se exatamente preto
        return (x * 255 + 0.5).astype(np.uint8)


def to_tensor01(img_u8: np.ndarray) -> torch.Tensor:
    return torch.from_numpy(np.ascontiguousarray(img_u8)).permute(2, 0, 1).float().div_(255.0)


class SegSet(torch.utils.data.Dataset):
    """Dataset de treino (anotado ou não). Devolve tensores em [0,1]; a normalização ImageNet faz parte do modelo."""

    def __init__(self, recs: list[dict], S: int, aug: str = "standard", labeled: bool = True):
        self.items = []
        for r in recs:
            g = letterbox_geom(*r["img"].shape[:2], S)
            c, m, v = apply_letterbox(r["img"], r["mask"] if labeled else None, g)
            self.items.append(dict(name=r["name"], img=c, mask=m, valid=v))
        self.aug, self.labeled, self._rng = Augmenter(aug), labeled, None

    def __len__(self) -> int:
        return len(self.items)

    def _get_rng(self) -> np.random.Generator:
        if self._rng is None:
            self._rng = np.random.default_rng((torch.initial_seed() + os.getpid()) % (2**32))
        return self._rng

    def __getitem__(self, i: int):
        it = self.items[i]
        img, mask, _ = self.aug(it["img"], it["mask"], it["valid"], self._get_rng())
        x = to_tensor01(img)
        if self.labeled:
            return x, torch.from_numpy(np.ascontiguousarray(mask)).long()
        return x

    def sample_weights(self, boost: float) -> np.ndarray:
        """w_i = 1 + boost * (fração de fibrina no 1º plano / média), limitado a [0, 2]. Só usa máscaras de TREINO."""
        fr = np.array([((it["mask"] == 1).sum() / max(1, (it["mask"] > 0).sum())) for it in self.items])
        return 1.0 + boost * np.clip(fr / max(fr.mean(), 1e-6), 0, 2.0)


class EvalItems:
    """Itens de avaliação: tela de entrada (sem augmentation) + geometria + GT no frame de avaliação (tamanho do conteúdo)."""

    def __init__(self, recs: list[dict], S: int, labeled: bool = True):
        self.names, self.geoms, self.gts, self.imgs, canv, cgt = [], [], [], [], [], []
        for r in recs:
            g = letterbox_geom(*r["img"].shape[:2], S)
            c, cm, _ = apply_letterbox(r["img"], r["mask"] if labeled else None, g)
            self.names.append(r["name"]); self.geoms.append(g); self.imgs.append(r["img"])
            self.gts.append(r["mask"] if labeled else None)
            canv.append(to_tensor01(c))
            if labeled:
                cgt.append(torch.from_numpy(cm).long())
        self.canvas = torch.stack(canv)
        self.canvas_gt = torch.stack(cgt) if labeled else None  # GT na tela do modelo (só para a perda de validação)

    def __len__(self) -> int:
        return len(self.names)


def worker_init(_):  # cada worker cria o seu gerador na primeira utilização (semente derivada de torch.initial_seed)
    cv2.setNumThreads(0)


def make_train_loader(train_set: SegSet, batch: int, boost: float, seed: int) -> torch.utils.data.DataLoader:
    w = torch.as_tensor(train_set.sample_weights(boost), dtype=torch.double)
    g = torch.Generator().manual_seed(seed)
    sampler = torch.utils.data.WeightedRandomSampler(w, num_samples=len(train_set), replacement=True, generator=g)
    nw = CFG["num_workers"]
    return torch.utils.data.DataLoader(train_set, batch_size=batch, sampler=sampler, num_workers=nw, drop_last=len(train_set) > batch,
                                       pin_memory=DEVICE.type == "cuda", persistent_workers=nw > 0, worker_init_fn=worker_init, generator=g)

In [ ]:
# Verificação visual do alinhamento imagem/máscara sob augmentation (as máscaras nunca são interpoladas)
_fr = get_frame(CFG["frame"])
_demo = SegSet(_fr["train"][:4], CFG["size"], aug="standard")
_demo._rng = np.random.default_rng(0)
fig, axes = plt.subplots(3, 6, figsize=(16, 8.3))
for j in range(6):
    x, m = _demo[j % 4]
    im = (x.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    axes[0, j].imshow(im); axes[1, j].imshow(colorize(m.numpy().astype(np.uint8))); axes[2, j].imshow(overlay(im, m.numpy().astype(np.uint8)))
    axes[0, j].set_title("augmentada", fontsize=8)
    for i in range(3):
        axes[i, j].axis("off")
fig.suptitle(f"Augmentation de treino (frame={CFG['frame']}, tela={CFG['size']}) — imagem, máscara, overlay")
fig.tight_layout(); savefig(fig, "aug_examples.png"); plt.show()
# teste automático: valores da máscara continuam em {0,1,2,3} após augmentation (sem interpolação de rótulos)
for _ in range(50):
    _x, _m = _demo[random.randrange(4)]
    assert set(torch.unique(_m).tolist()) <= {0, 1, 2, 3}
print("OK: máscaras augmentadas só contêm rótulos 0..3")

## 10. Arquiteturas e pesos pré-treinados

Três configurações complementares (relação desempenho/estabilidade/custo), todas com normalização ImageNet embutida no modelo:

| id | Arquitetura | Encoder (pesos) | Papel |
|---|---|---|---|
| `unetpp_effb4` | U-Net++ (`smp`) | EfficientNet-B4 `ra2_in1k` (timm) | CNN eficiente, *skip connections* densas — **referência** |
| `upernet_convnext_s` | UPerNet (`smp`) | ConvNeXt-S `fb_in22k_ft_in1k` (timm) | CNN moderna com pirâmide multiescala/contexto global |
| `segformer_mitb3` | SegFormer (`smp`) | MiT-B3 ImageNet-1k (NVlabs, via `smp`) | Transformer hierárquico (encoder do artigo, mas **sem** P-scSE) |

**Origem dos pesos.** `build_model` tenta, por esta ordem: (1) descarregar pesos pré-treinados (precisa de Internet); (2) carregar um
ficheiro local (`.pth/.pt/.bin/.safetensors`) encontrado em `/kaggle/input` cujo nome/pasta contenha a chave do encoder
(p.ex. `efficientnet_b4`, `convnext_small`, `mit_b3`); (3) **pesos aleatórios, com aviso e registo**. A origem de cada execução é
gravada em `weights_provenance.json` (inclui `source` e a nota de licença **a confirmar no *model card***). Os pesos MiT do
SegFormer têm licença da NVIDIA (uso não comercial); confirme as licenças dos restantes antes de reutilizar.

In [ ]:
ARCH_SPECS = {
    "unetpp_effb4": dict(
        ctor=lambda w: smp.UnetPlusPlus("tu-efficientnet_b4.ra2_in1k", encoder_weights=w, classes=K, in_channels=3),
        key="efficientnet_b4", family="CNN", origin="timm efficientnet_b4.ra2_in1k (ImageNet-1k, RandAugment recipe)",
        license="ver model card timm/efficientnet_b4.ra2_in1k (Apache-2.0 declarado no código timm; confirmar nos pesos)"),
    "upernet_convnext_s": dict(
        ctor=lambda w: smp.UPerNet("tu-convnext_small.fb_in22k_ft_in1k", encoder_weights=w, classes=K, in_channels=3),
        key="convnext_small", family="CNN", origin="timm convnext_small.fb_in22k_ft_in1k (ImageNet-22k -> 1k, Meta AI)",
        license="ver model card timm/convnext_small.fb_in22k_ft_in1k (código ConvNeXt: MIT; confirmar nos pesos)"),
    "segformer_mitb3": dict(
        ctor=lambda w: smp.Segformer("mit_b3", encoder_weights=w, classes=K, in_channels=3),
        key="mit_b3", family="Transformer", origin="NVlabs SegFormer MiT-B3 ImageNet-1k (distribuído por smp / HF)",
        license="NVIDIA Source Code License (não comercial) — confirmar antes de uso comercial"),
    "smoke_fpn_r18": dict(  # apenas para o modo smoke (teste de pipeline)
        ctor=lambda w: smp.FPN("resnet18", encoder_weights=w, classes=K, in_channels=3),
        key="resnet18", family="CNN", origin="torchvision resnet18 (ImageNet-1k)", license="BSD-3 (torchvision)"),
}
WEIGHTS_PROVENANCE: dict[str, dict] = {}
_WARNED_RANDOM: set[str] = set()


class Net(nn.Module):
    """Entrada em [0,1]; normalização ImageNet interna; saída: *logits* (B,K,H,W)."""

    def __init__(self, core: nn.Module):
        super().__init__()
        self.core = core
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.core((x - self.mean) / self.std)

    def encoder_params(self):
        return list(self.core.encoder.parameters())

    def other_params(self):
        enc = {id(p) for p in self.core.encoder.parameters()}
        return [p for p in self.parameters() if id(p) not in enc]


def _local_weight_files(key: str, max_depth: int = 5) -> list[Path]:
    out = []
    if INPUT_ROOT.exists():
        d0 = len(INPUT_ROOT.parts)
        for dp, dn, fn in os.walk(INPUT_ROOT):
            if len(Path(dp).parts) - d0 >= max_depth:
                dn[:] = []
            for f in fn:
                if f.lower().endswith((".pth", ".pt", ".bin", ".safetensors")) and key in (dp + "/" + f).lower():
                    out.append(Path(dp) / f)
    return out


def _read_state(path: Path) -> dict:
    if path.suffix == ".safetensors":
        from safetensors.torch import load_file
        return load_file(str(path))
    sd = torch.load(path, map_location="cpu", weights_only=True)
    for k in ("state_dict", "model", "state"):
        if isinstance(sd, dict) and k in sd and isinstance(sd[k], dict):
            sd = sd[k]
    return {k.replace("module.", "", 1): v for k, v in sd.items()}


def _match_ratio(sd: dict, target: dict) -> tuple[dict, float]:
    """Casa chaves (também com 'a.b.c' -> 'a_b.c', como no `features_only` do timm) e devolve fração dos parâmetros-alvo cobertos."""
    matched = {}
    for k, v in sd.items():
        cands = [k] + [k.replace(".", "_", n) for n in (1, 2, 3)]
        for c in cands:
            if c in target and target[c].shape == v.shape:
                matched[c] = v
                break
    tot = sum(t.numel() for t in target.values())
    cov = sum(t.numel() for k, t in target.items() if k in matched)
    return matched, cov / max(tot, 1)


def load_local_encoder(core: nn.Module, path: Path) -> float:
    sd = _read_state(path)
    best = (None, None, 0.0)
    for tgt in [getattr(core.encoder, "model", None), core.encoder]:
        if tgt is None:
            continue
        m, r = _match_ratio(sd, tgt.state_dict())
        if r > best[2]:
            best = (tgt, m, r)
    if best[0] is not None and best[2] >= 0.8:
        best[0].load_state_dict(best[1], strict=False)
    return best[2]


def build_model(arch: str, pretrained: bool = True) -> tuple[Net, dict]:
    spec = ARCH_SPECS[arch]
    prov = dict(arch=arch, family=spec["family"], origin=spec["origin"], license_note=spec["license"], requested_pretrained=pretrained, source="random_init")
    core = None
    if pretrained:
        try:
            core = spec["ctor"]("imagenet")
            prov["source"] = "downloaded_or_cached_pretrained"
        except Exception as e:  # noqa: BLE001 — sem Internet / sem cache
            prov["online_error"] = f"{type(e).__name__}: {str(e)[:160]}"
            files = _local_weight_files(spec["key"])
            for f in files:
                core = spec["ctor"](None)
                ratio = load_local_encoder(core, f)
                if ratio >= 0.8:
                    prov.update(source=f"local_file:{f}", local_match_ratio=round(ratio, 3))
                    break
                core = None
            if core is None and arch not in _WARNED_RANDOM:
                _WARNED_RANDOM.add(arch)
                print(f"AVISO [{arch}]: pesos pré-treinados indisponíveis (Internet off e sem ficheiro local). A usar INICIALIZAÇÃO ALEATÓRIA — "
                      f"o desempenho será pior e a comparação não é válida como 'transfer learning'.")
    if core is None:
        core = spec["ctor"](None)
        prov["pretrained_effective"] = False
    else:
        prov["pretrained_effective"] = True
    WEIGHTS_PROVENANCE[arch] = prov
    return Net(core), prov


def count_params(m: nn.Module) -> float:
    return sum(p.numel() for p in m.parameters()) / 1e6

## 11. Funções de perda
`ce_dice` (padrão): *cross-entropy* ponderada por classe (pesos ∝ 1/√frequência, **limitados** a `class_weight_cap` para não inflacionar falsos positivos)
+ Dice multiclasse sobre os 3 tecidos. Alternativas comparadas na validação em modo `full`: `focal_dice`, `focal_tversky`
(Tversky α=0,3, β=0,7, γ=0,75 + 0,5·CE) e `ce_dice_lovasz`. `ignore_index=255` suporta *pseudo-labels* incertos no SSL.

In [ ]:
IGNORE = 255


def class_weights_from(train_recs: list[dict], cap: float) -> torch.Tensor:
    cnt = np.zeros(K)
    for r in train_recs:
        cnt += np.bincount(r["mask"].ravel(), minlength=K)[:K]
    freq = cnt / cnt.sum()
    w = 1.0 / np.sqrt(np.maximum(freq, 1e-8))
    w = np.minimum(w / w.min(), cap)
    return torch.tensor(w / w.mean(), dtype=torch.float32)


class SegLoss(nn.Module):
    def __init__(self, name: str, class_w: torch.Tensor | None, label_smoothing: float = 0.0):
        super().__init__()
        self.name = name
        self.register_buffer("cw", class_w if class_w is not None else torch.ones(K))
        self.ls = label_smoothing
        self.dice = smp.losses.DiceLoss("multiclass", classes=TISSUES, from_logits=True, ignore_index=IGNORE)
        self.focal = smp.losses.FocalLoss("multiclass", gamma=2.0, ignore_index=IGNORE)
        self.tversky = smp.losses.TverskyLoss("multiclass", classes=TISSUES, from_logits=True, alpha=0.3, beta=0.7, gamma=0.75, ignore_index=IGNORE)
        self.lovasz = smp.losses.LovaszLoss("multiclass", per_image=False, from_logits=True, ignore_index=IGNORE)
        assert name in ("ce", "ce_dice", "focal_dice", "focal_tversky", "ce_dice_lovasz"), name

    def ce(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.cw, ignore_index=IGNORE, label_smoothing=self.ls)

    def forward(self, logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        logits = logits.float()
        n = self.name
        if n == "ce":
            return self.ce(logits, y)
        if n == "ce_dice":
            return self.ce(logits, y) + self.dice(logits, y)
        if n == "focal_dice":
            return self.focal(logits, y) + self.dice(logits, y)
        if n == "focal_tversky":
            return self.tversky(logits, y) + 0.5 * self.ce(logits, y)
        return self.ce(logits, y) + self.dice(logits, y) + 0.5 * self.lovasz(logits, y)


def _loss_selftest():
    torch.manual_seed(0)
    lg = torch.randn(2, K, 32, 32, requires_grad=True)
    y = torch.randint(0, K, (2, 32, 32))
    y[0, :8] = IGNORE
    cw = torch.tensor([1.0, 3.0, 2.0, 1.5])
    for n in ["ce", "ce_dice", "focal_dice", "focal_tversky", "ce_dice_lovasz"]:
        l = SegLoss(n, cw)(lg, y)
        l.backward()
        assert torch.isfinite(l) and lg.grad is not None and torch.isfinite(lg.grad).all(), n
        lg.grad = None
    # perda ~0 quando o logit aponta com muita confiança para a classe certa
    y2 = torch.randint(0, K, (2, 32, 32))
    perfect = F.one_hot(y2, K).permute(0, 3, 1, 2).float() * 30
    assert SegLoss("ce_dice", None)(perfect, y2) < 0.05
    print("OK: testes das funções de perda (finitas, gradiente, ignore_index, ~0 no caso perfeito)")


_loss_selftest()

## 12. Métricas e testes automáticos

Tudo é calculado a partir de **matrizes de confusão por imagem** (linhas = GT, colunas = predição), que permitem agregar sobre o dataset e fazer
*bootstrap* sobre imagens. Definições (fundo explicitamente incluído/excluído):

| Nome | Definição | Fundo |
|---|---|---|
| `dice_<classe>` | Dice da classe sobre a matriz de confusão **somada** no conjunto | — |
| `macro_fg_dice` | média dos 3 Dice de tecido | **excluído** (métrica de seleção) |
| `macro_all_dice` | média dos 4 Dice | incluído |
| `micro_fg_dice` | 2ΣTP/(2ΣTP+ΣFP+ΣFN) sobre os 3 tecidos (confusões entre tecidos contam como FP *e* FN) | excluído |
| `binary_fg_dice` | ferida (qualquer tecido) vs fundo | — |
| `imgmean_dice_<classe>` | média por imagem, só onde a classe existe no GT ou na predição | — |
| `paper_*` | **réplica do protocolo do código dos autores** (acumuladores não repostos entre classes; fundo excluído). É a única candidata a comparação direta com 87,64 | excluído |

Os testes abaixo cobrem casos conhecidos, máscaras vazias, classes ausentes, desalinhamento e **concordância exata** da réplica
vetorizada com uma porta literal do ciclo original (conjuntos de coordenadas) em casos aleatórios.

In [ ]:
def confusion_matrix_np(gt: np.ndarray, pred: np.ndarray) -> np.ndarray:
    if gt.shape != pred.shape:
        raise ValueError(f"Predição {pred.shape} desalinhada com o GT {gt.shape}")
    return np.bincount(gt.astype(np.int64).ravel() * K + pred.astype(np.int64).ravel(), minlength=K * K).reshape(K, K)


def _div(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.where(b > 0, a / np.where(b > 0, b, 1), np.nan)


def per_class_from_cm(cm: np.ndarray) -> dict:
    tp = np.diag(cm).astype(float)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    return dict(tp=tp, fp=fp, fn=fn, dice=_div(2 * tp, 2 * tp + fp + fn), iou=_div(tp, tp + fp + fn), precision=_div(tp, tp + fp), recall=_div(tp, tp + fn))


def paper_image_stats(cm: np.ndarray, ep: float = 1e-6) -> dict:
    """Réplica do ciclo por imagem do código dos autores (supervised_training.ipynb): acumuladores tp/fp/fn por imagem, **não repostos entre classes**,
    somados ao total global após cada classe comum; classes só em GT -> FN; só na predição -> FP; ausentes em ambas são ignoradas."""
    tp = fp = fn = 0.0
    c_tp = c_fp = c_fn = 0.0
    per = {}
    lg = {c for c in TISSUES if cm[c].sum() > 0}
    lp = {c for c in TISSUES if cm[:, c].sum() > 0}
    for d in sorted(lg - lp):
        fn += cm[d].sum()
        c_fn += fn
    for d in sorted(lp - lg):
        fp += cm[:, d].sum()
        c_fp += fp
    for d in sorted(lg ^ lp):
        per[d] = (0.0, 0.0, 0.0, 0.0)
    for c in sorted(lg & lp):
        t, f_p, f_n = cm[c, c], cm[:, c].sum() - cm[c, c], cm[c].sum() - cm[c, c]
        tp, fp, fn = tp + t, fp + f_p, fn + f_n
        c_tp, c_fp, c_fn = c_tp + tp, c_fp + fp, c_fn + fn
        per[c] = (tp / (tp + fp + ep) * 100, tp / (tp + fn + ep) * 100, 2 * tp / (2 * tp + fp + fn + ep) * 100, tp / (tp + fp + fn + ep) * 100)
    return dict(c=np.array([c_tp, c_fp, c_fn]), per=per)


def _paper_reference_literal(gt: np.ndarray, pred: np.ndarray, ep: float = 1e-6):
    """Porta LITERAL do ciclo original (conjuntos de coordenadas) — usada só para testar a versão vetorizada."""
    stp = sfp = sfn = 0
    tp = fp = fn = 0
    lbl_gt, lbl_pred = set(np.unique(gt)), set(np.unique(pred))
    lbl_gt.discard(0); lbl_pred.discard(0)
    diff1, diff2 = lbl_gt - lbl_pred, lbl_pred - lbl_gt
    diffs = diff1.union(diff2)
    per = {}
    for d1 in diff1:
        fn += len(np.argwhere(gt == d1)); sfn += fn
    for d2 in diff2:
        fp += len(np.argwhere(pred == d2)); sfp += fp
    for d in diffs:
        per[int(d)] = (0.0, 0.0, 0.0, 0.0)
    for cmn in lbl_gt.intersection(lbl_pred):
        gi, pi = np.where(gt == cmn), np.where(pred == cmn)
        g_t = tuple((gi[0][i], gi[1][i]) for i in range(len(gi[0])))
        p_t = tuple((pi[0][i], pi[1][i]) for i in range(len(pi[0])))
        tp += len(set(g_t) & set(p_t)); fp += len(set(p_t) - set(g_t)); fn += len(set(g_t) - set(p_t))
        stp += tp; sfp += fp; sfn += fn
        per[int(cmn)] = (tp / (tp + fp + ep) * 100, tp / (tp + fn + ep) * 100, 2 * tp / (2 * tp + fp + fn + ep) * 100, tp / (tp + fp + fn + ep) * 100)
    return np.array([stp, sfp, sfn], float), per


class EvalResult:
    """Resultado de avaliação sobre N imagens: guarda matrizes de confusão por imagem e estatísticas do protocolo-réplica."""

    def __init__(self, names: list[str], cms: np.ndarray):
        self.names, self.cms = list(names), np.asarray(cms, dtype=np.int64)
        ps = [paper_image_stats(cm) for cm in self.cms]
        self.paper_c = np.stack([p["c"] for p in ps])                      # (N,3): contribuições a stp, sfp, sfn
        self.paper_per = np.full((len(ps), K, 4), np.nan)                  # (N,K,4): P,R,Dice,IoU por classe (nan = não avaliada)
        for i, p in enumerate(ps):
            for c, v in p["per"].items():
                self.paper_per[i, c] = v

    def __len__(self):
        return len(self.names)

    def metrics(self, idx=None) -> dict:
        idx = np.arange(len(self)) if idx is None else np.asarray(idx)
        cm = self.cms[idx].sum(0)
        pc = per_class_from_cm(cm)
        fg = TISSUES
        tp, fp, fn = pc["tp"][fg].sum(), pc["fp"][fg].sum(), pc["fn"][fg].sum()
        tb, fb, nb = cm[1:, 1:].sum(), cm[0, 1:].sum(), cm[1:, 0].sum()
        c = self.paper_c[idx].sum(0)
        pp = self.paper_per[idx]
        out = dict(
            macro_fg_dice=np.nanmean(pc["dice"][fg]) if not np.isnan(pc["dice"][fg]).all() else np.nan,
            macro_all_dice=np.nanmean(pc["dice"]) if not np.isnan(pc["dice"]).all() else np.nan,
            macro_fg_iou=np.nanmean(pc["iou"][fg]) if not np.isnan(pc["iou"][fg]).all() else np.nan,
            macro_all_iou=np.nanmean(pc["iou"]) if not np.isnan(pc["iou"]).all() else np.nan,
            macro_fg_precision=np.nanmean(pc["precision"][fg]) if not np.isnan(pc["precision"][fg]).all() else np.nan,
            macro_fg_recall=np.nanmean(pc["recall"][fg]) if not np.isnan(pc["recall"][fg]).all() else np.nan,
            micro_fg_dice=_div(2 * tp, 2 * tp + fp + fn), micro_fg_iou=_div(tp, tp + fp + fn),
            micro_fg_precision=_div(tp, tp + fp), micro_fg_recall=_div(tp, tp + fn),
            binary_fg_dice=_div(2 * tb, 2 * tb + fb + nb), pixel_acc=_div(np.trace(cm), cm.sum()),
            paper_dice=_div(2 * c[0], 2 * c[0] + c[1] + c[2]), paper_iou=_div(c[0], c.sum()),
            paper_precision=_div(c[0], c[0] + c[1]), paper_recall=_div(c[0], c[0] + c[2]),
        )
        out = {k: float(v) for k, v in out.items()}
        for k in range(K):
            n = CLASS_NAMES[k].lower()
            for key in ("dice", "iou", "precision", "recall"):
                out[f"{key}_{n}"] = float(pc[key][k])
            if k in fg:
                col = pp[:, k, :]
                ok = ~np.isnan(col[:, 2])
                for j, key in enumerate(("precision", "recall", "dice", "iou")):
                    out[f"paper_{key}_{n}"] = float(col[ok, j].mean() / 100) if ok.any() else np.nan
        # média por imagem (só onde a classe existe em GT ou predição)
        for k in fg:
            d = np.array([per_class_from_cm(m)["dice"][k] for m in self.cms[idx]])
            out[f"imgmean_dice_{CLASS_NAMES[k].lower()}"] = float(np.nanmean(d)) if (~np.isnan(d)).any() else np.nan
        return out

    def bootstrap(self, keys: list[str], B: int, seed: int = 0) -> dict:
        rng = np.random.default_rng(seed)
        draws = {k: [] for k in keys}
        n = len(self)
        for _ in range(B):
            m = self.metrics(rng.integers(0, n, n))
            for k in keys:
                draws[k].append(m[k])
        return {k: tuple(np.nanpercentile(v, [2.5, 97.5])) for k, v in draws.items()}


def evaluate_preds(names: list[str], gts: list[np.ndarray], preds: list[np.ndarray]) -> EvalResult:
    if not (len(names) == len(gts) == len(preds)):
        raise ValueError("Número de GT/predições/nomes inconsistente")
    return EvalResult(names, np.stack([confusion_matrix_np(g, p) for g, p in zip(gts, preds)]))


def run_metric_tests() -> None:
    rng = np.random.default_rng(123)
    # 1) exemplo conhecido
    gt = np.array([[0, 1], [2, 2]]); pr = np.array([[0, 1], [2, 1]])
    m = evaluate_preds(["a"], [gt], [pr]).metrics()
    assert abs(m["dice_fibrin"] - 2 / 3) < 1e-9 and abs(m["dice_granulation"] - 2 / 3) < 1e-9 and np.isnan(m["dice_callus"])
    assert abs(m["macro_fg_dice"] - 2 / 3) < 1e-9 and abs(m["micro_fg_dice"] - 2 / 3) < 1e-9
    # 2) predição perfeita -> 1 em tudo
    g = rng.integers(0, 4, (30, 30)); m = evaluate_preds(["p"], [g], [g.copy()]).metrics()
    assert m["macro_fg_dice"] == 1 and m["micro_fg_dice"] == 1 and m["paper_dice"] == 1 and m["pixel_acc"] == 1
    # 3) máscaras vazias (só fundo) e predição vazia: classes ausentes em ambos -> NaN, sem divisão por zero/exceção
    z = np.zeros((16, 16), int); m = evaluate_preds(["e"], [z], [z]).metrics()
    assert np.isnan(m["macro_fg_dice"]) and np.isnan(m["paper_dice"]) and m["pixel_acc"] == 1
    # 4) classe só no GT (não prevista) -> Dice 0; só na predição -> Dice 0 (FP)
    gt = np.zeros((10, 10), int); gt[:5] = 1; m = evaluate_preds(["f"], [gt], [np.zeros_like(gt)]).metrics()
    assert m["dice_fibrin"] == 0 and m["recall_fibrin"] == 0
    m = evaluate_preds(["g"], [np.zeros_like(gt)], [gt]).metrics()
    assert m["dice_fibrin"] == 0 and m["precision_fibrin"] == 0
    # 5) desalinhamento deve falhar de forma segura
    try:
        confusion_matrix_np(np.zeros((4, 4), int), np.zeros((4, 5), int)); raise AssertionError("devia falhar")
    except ValueError:
        pass
    # 6) concordância com sklearn
    from sklearn.metrics import confusion_matrix as sk_cm
    g, p = rng.integers(0, 4, (50, 50)), rng.integers(0, 4, (50, 50))
    assert (confusion_matrix_np(g, p) == sk_cm(g.ravel(), p.ravel(), labels=range(K))).all()
    # 7) réplica vetorizada == porta literal do ciclo original (inclui classes ausentes/só-GT/só-predição)
    for t in range(150):
        g = rng.integers(0, 4, (12, 12)) * (rng.random((12, 12)) < rng.uniform(0.2, 1))
        p = np.where(rng.random((12, 12)) < rng.uniform(0.3, 1), g, rng.integers(0, 4, (12, 12)))
        if t % 5 == 0:
            g[g == 2] = 0
        lit_c, lit_per = _paper_reference_literal(g, p)
        vec = paper_image_stats(confusion_matrix_np(g, p))
        assert np.allclose(lit_c, vec["c"]), (t, lit_c, vec["c"])
        assert lit_per.keys() == vec["per"].keys() and all(np.allclose(lit_per[k], vec["per"][k]) for k in lit_per), (t, lit_per, vec["per"])
    # 8) bootstrap: IC contém a estimativa pontual em dados simulados
    g = [rng.integers(0, 4, (20, 20)) for _ in range(12)]; p = [np.where(rng.random((20, 20)) < 0.7, x, rng.integers(0, 4, (20, 20))) for x in g]
    r = evaluate_preds([str(i) for i in range(12)], g, p); ci = r.bootstrap(["macro_fg_dice"], 200)["macro_fg_dice"]
    assert ci[0] <= r.metrics()["macro_fg_dice"] <= ci[1]
    print("OK: testes de métricas (exemplo conhecido, perfeito, vazias, ausentes, desalinhamento, sklearn, réplica==porta literal em 150 casos, bootstrap)")


run_metric_tests()

## 13. Motor de treino e validação

AdamW com **LR diferencial** (encoder ≪ decoder), *warmup* linear + cosseno, **AMP fp16** com `GradScaler`, *gradient clipping*,
**EMA** dos pesos (o modelo avaliado e guardado é o EMA), *freeze* inicial do encoder pré-treinado (*fine-tuning* progressivo),
*early stopping* por `macro_fg_dice` de validação, *checkpointing* com **retoma** (`last.pt`) e sondagem automática de OOM
(reduz o micro-batch e compensa com acumulação, mantendo o batch efetivo). A validação é calculada no **frame de avaliação**
(tamanho original do conteúdo), com probabilidades revertidas por interpolação bilinear.

In [ ]:
AUTOCAST = lambda: torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)  # noqa: E731
TTA_SETS = {
    "none": [(0, 0, 1.0)],
    "flip": [(h, v, 1.0) for h in (0, 1) for v in (0, 1)],
    "flip_ms": [(h, v, s) for h in (0, 1) for v in (0, 1) for s in (0.85, 1.0, 1.2)],
}


class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.steps = decay, 0

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        self.steps += 1
        d = min(self.decay, (1 + self.steps) / (10 + self.steps))  # warm-up do decay
        for e, p in zip(self.module.parameters(), model.parameters()):
            e.mul_(d).add_(p.detach(), alpha=1 - d)
        for eb, b in zip(self.module.buffers(), model.buffers()):
            eb.copy_(b)


@torch.no_grad()
def canvas_probs(model: nn.Module, x: torch.Tensor, tta: str) -> torch.Tensor:
    """Média das probabilidades softmax sobre as transformações reversíveis do TTA. x: (B,3,S,S) em [0,1] -> (B,K,S,S)."""
    S, acc, n = x.shape[-1], 0, 0
    for fh, fv, s in TTA_SETS[tta]:
        xi = x
        if fh:
            xi = xi.flip(3)
        if fv:
            xi = xi.flip(2)
        side = max(32, int(round(S * s / 32)) * 32)
        if side != S:
            xi = F.interpolate(xi, size=(side, side), mode="bilinear", align_corners=False)
        with AUTOCAST():
            lg = model(xi)
        p = lg.float().softmax(1)
        if side != S:
            p = F.interpolate(p, size=(S, S), mode="bilinear", align_corners=False)
        if fv:
            p = p.flip(2)
        if fh:
            p = p.flip(3)
        acc, n = acc + p, n + 1
    return acc / n


@torch.no_grad()
def predict_probs(models: list[nn.Module], items: EvalItems, tta: str = "none", weights: list[float] | None = None, bs: int = 8) -> list[np.ndarray]:
    """Probabilidades (K,h,w) float32 por imagem, no frame de avaliação, médias (ponderadas) sobre modelos e TTA."""
    weights = weights or [1.0] * len(models)
    wsum = float(sum(weights))
    out = []
    for m in models:
        m.eval()
    for i in range(0, len(items), bs):
        x = items.canvas[i:i + bs].to(DEVICE, non_blocking=True)
        p = sum(w * canvas_probs(m, x, tta) for m, w in zip(models, weights)) / wsum
        for j in range(p.shape[0]):
            out.append(unletterbox_probs(p[j].cpu(), items.geoms[i + j]).numpy().astype(np.float32))
    return out


def remove_small_components(pred: np.ndarray, min_area_frac: float) -> np.ndarray:
    """Remove componentes conexas de tecido com área < frac da área da imagem; o rótulo de substituição é a moda dos vizinhos (ou fundo)."""
    if min_area_frac <= 0:
        return pred
    out, thr = pred.copy(), min_area_frac * pred.size
    for c in TISSUES:
        n, lab, stats, _ = cv2.connectedComponentsWithStats((out == c).astype(np.uint8), connectivity=8)
        for comp in range(1, n):
            if stats[comp, cv2.CC_STAT_AREA] < thr:
                region = lab == comp
                ring = cv2.dilate(region.astype(np.uint8), np.ones((3, 3), np.uint8), iterations=2).astype(bool) & ~region
                vals = out[ring & (out != c)]
                out[region] = np.bincount(vals, minlength=K).argmax() if vals.size else 0
    return out


def probs_to_preds(probs: list[np.ndarray], min_area_frac: float = 0.0) -> list[np.ndarray]:
    return [remove_small_components(p.argmax(0).astype(np.uint8), min_area_frac) for p in probs]


def evaluate_models(models, items: EvalItems, tta: str = "none", weights=None, min_area_frac: float = 0.0):
    probs = predict_probs(models, items, tta, weights)
    return evaluate_preds(items.names, items.gts, probs_to_preds(probs, min_area_frac)), probs


@torch.no_grad()
def canvas_loss(model: nn.Module, items: EvalItems, criterion: nn.Module, bs: int = 8) -> float:
    model.eval()
    tot, n = 0.0, 0
    for i in range(0, len(items), bs):
        x, y = items.canvas[i:i + bs].to(DEVICE), items.canvas_gt[i:i + bs].to(DEVICE)
        with AUTOCAST():
            lg = model(x)
        tot += float(criterion(lg.float(), y)) * len(x)
        n += len(x)
    return tot / max(n, 1)


def set_encoder_trainable(model: Net, flag: bool) -> None:
    for p in model.encoder_params():
        p.requires_grad_(flag)


def probe_micro_batch(model: Net, S: int, target: int) -> int:
    """Maior micro-batch (potência de 2 <= target) que executa fwd+bwd sem OOM. Em CPU devolve `target`."""
    if DEVICE.type != "cuda":
        return target
    backup = copy.deepcopy(model.state_dict())
    bs = target
    while bs >= 1:
        try:
            model.train()
            x = torch.rand(bs, 3, S, S, device=DEVICE)
            with AUTOCAST():
                lg = model(x)
            lg.float().mean().backward()
            model.zero_grad(set_to_none=True)
            model.load_state_dict(backup)
            torch.cuda.empty_cache()
            return bs
        except torch.cuda.OutOfMemoryError:
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            bs //= 2
    raise RuntimeError("Nem micro-batch 1 cabe na GPU: reduza `size`.")


def lr_factor(step: int, warm: int, total: int, min_ratio: float) -> float:
    if step < warm:
        return (step + 1) / max(warm, 1)
    t = (step - warm) / max(total - warm, 1)
    return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * min(t, 1.0)))


def run_id_of(spec: dict) -> str:
    h = hashlib.md5(json.dumps(spec, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{spec['exp']}__{spec['arch']}__s{spec['seed']}__{h}"


def default_spec(**over) -> dict:
    s = dict(exp="sup", arch=CFG["archs"][0], seed=CFG["seeds"][0], frame=CFG["frame"], size=CFG["size"], loss=CFG["loss"], epochs=CFG["epochs"],
             patience=CFG["patience"], pretrained=True, aug="standard", sampler=True, lr_decoder=CFG["lr_decoder"], lr_encoder=CFG["lr_encoder"],
             freeze_epochs=CFG["freeze_epochs"], batch_size=CFG["batch_size"], keep_ckpt=True)
    s.update(over)
    return s


def _save_half(state: dict, path: Path) -> None:
    torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in state.items()}, path)


def _load_half(path: Path) -> dict:
    sd = torch.load(path, map_location="cpu", weights_only=True)
    return {k: (v.float() if v.is_floating_point() else v) for k, v in sd.items()}


def _train_run(spec: dict, micro_bs_hint: int | None = None) -> dict:
    rid = run_id_of(spec)
    rdir = DIRS["runs"] / rid
    rdir.mkdir(parents=True, exist_ok=True)
    seed_everything(spec["seed"])
    S = spec["size"]
    recs = get_frame(spec["frame"])
    train_set = SegSet(recs["train"], S, aug=spec["aug"])
    val_items = EvalItems(recs["val"], S)
    loader = make_train_loader(train_set, spec["batch_size"], CFG["fibrin_boost"] if spec["sampler"] else 0.0, spec["seed"])
    model, prov = build_model(spec["arch"], spec["pretrained"])
    model.to(DEVICE)
    micro_bs = micro_bs_hint or probe_micro_batch(model, S, spec["batch_size"])
    criterion = SegLoss(spec["loss"], class_weights_from(recs["train"], CFG["class_weight_cap"]), CFG["label_smoothing"]).to(DEVICE)
    opt = torch.optim.AdamW([{"params": model.encoder_params(), "lr": spec["lr_encoder"]}, {"params": model.other_params(), "lr": spec["lr_decoder"]}],
                            weight_decay=CFG["weight_decay"])
    spe = len(loader)
    total, warm = spec["epochs"] * spe, CFG["warmup_epochs"] * spe
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda st: lr_factor(st, warm, total, CFG["min_lr_ratio"]))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    ema = EMA(model, CFG["ema_decay"])
    hist, best, best_epoch, bad, start = [], -1.0, -1, 0, 0

    last = rdir / "last.pt"
    if last.exists() and CFG["resume"]:
        ck = torch.load(last, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); ema.module.load_state_dict(ck["ema"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"]); ema.steps = ck["ema_steps"]
        hist, best, best_epoch, bad, start = ck["hist"], ck["best"], ck["best_epoch"], ck["bad"], ck["epoch"] + 1
        print(f"  [retoma] {rid} a partir da época {start}")

    t0, status = time.time(), "finished"
    for epoch in range(start, spec["epochs"]):
        if not BUDGET.ok(30):
            status = "stopped_by_time_budget"
            break
        freeze = spec["pretrained"] and epoch < spec["freeze_epochs"]
        set_encoder_trainable(model, not freeze)
        model.train()
        run_loss, nb = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            while True:  # repete o batch com micro-batch menor se houver OOM
                try:
                    opt.zero_grad(set_to_none=True)
                    loss_tot = 0.0
                    for xc, yc in zip(xb.split(micro_bs), yb.split(micro_bs)):
                        with AUTOCAST():
                            lg = model(xc)
                        loss = criterion(lg.float(), yc) * (len(xc) / len(xb))
                        scaler.scale(loss).backward()
                        loss_tot += float(loss)
                    break
                except torch.cuda.OutOfMemoryError:
                    opt.zero_grad(set_to_none=True)
                    torch.cuda.empty_cache()
                    if micro_bs == 1:
                        raise
                    micro_bs //= 2
                    print(f"  OOM -> micro-batch {micro_bs}")
            if not math.isfinite(loss_tot):
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
            run_loss, nb = run_loss + loss_tot, nb + 1
        res, _ = evaluate_models([ema.module], val_items, "none")
        vm = res.metrics()
        score = 0.0 if np.isnan(vm["macro_fg_dice"]) else vm["macro_fg_dice"]
        hist.append(dict(epoch=epoch, train_loss=run_loss / max(nb, 1), val_loss=canvas_loss(ema.module, val_items, criterion),
                         val_macro_fg_dice=vm["macro_fg_dice"], val_micro_fg_dice=vm["micro_fg_dice"], val_macro_fg_iou=vm["macro_fg_iou"],
                         val_paper_dice=vm["paper_dice"], val_dice_fibrin=vm["dice_fibrin"], val_dice_granulation=vm["dice_granulation"],
                         val_dice_callus=vm["dice_callus"], lr_dec=opt.param_groups[1]["lr"]))
        if score > best + 1e-4:
            best, best_epoch, bad = score, epoch, 0
            _save_half(ema.module.state_dict(), rdir / "best.pt")
        else:
            bad += 1
        if (epoch + 1) % 5 == 0:  # ponto de retoma (best.pt é gravado sempre que a validação melhora)
            torch.save(dict(model=model.state_dict(), ema=ema.module.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                            ema_steps=ema.steps, hist=hist, best=best, best_epoch=best_epoch, bad=bad, epoch=epoch), last)
        if epoch % max(1, spec["epochs"] // 10) == 0 or epoch == spec["epochs"] - 1:
            h = hist[-1]
            print(f"  {rid} ep{epoch:03d} loss {h['train_loss']:.3f}/{h['val_loss']:.3f} valDice(macro-fg) {h['val_macro_fg_dice']:.4f} best {best:.4f}@{best_epoch}")
        if bad >= spec["patience"]:
            status = "early_stopped"
            break

    # reavalia o MELHOR checkpoint carregado de disco (fp16->fp32), para que o resultado seja o do artefacto guardado
    final = Net(ARCH_SPECS[spec["arch"]]["ctor"](None)).to(DEVICE)
    final.load_state_dict(_load_half(rdir / "best.pt"))
    res, _ = evaluate_models([final], val_items, "none")
    result = dict(spec=spec, run_id=rid, status=status, best_epoch=best_epoch, epochs_run=len(hist), val=res.metrics(), history=hist, provenance=prov,
                  params_M=round(count_params(model), 2), micro_batch=micro_bs, train_seconds=round(time.time() - t0, 1), ckpt=str(rdir / "best.pt"),
                  device=HW["gpu_name"] or "cpu")
    (rdir / "result.json").write_text(json.dumps(result, indent=1, default=float))
    if last.exists():
        last.unlink()
    if not spec["keep_ckpt"]:
        (rdir / "best.pt").unlink(missing_ok=True)
        result["ckpt"] = None
    del model, ema, opt, final
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result


def train_run(spec: dict) -> dict:
    """Treina (ou reutiliza, se já existir `result.json`) uma execução. Idempotente: permite reiniciar a sessão do notebook."""
    rid = run_id_of(spec)
    rf = DIRS["runs"] / rid / "result.json"
    if rf.exists() and CFG["resume"]:
        print(f"[reutilizado] {rid}")
        return json.loads(rf.read_text())
    print(f"[treino] {rid}")
    return _train_run(spec)

## 14a. Verificação prévia: o modelo consegue memorizar? (sanity check) e *LR range test*
Antes de gastar horas, confirma-se que o pipeline (dados→modelo→perda→métricas) **consegue sobreajustar 8 imagens sem augmentation**.
Se não conseguir, há um erro de pipeline, não de hiperparâmetros. O *LR range test* é só consultivo (não altera `CFG`).

In [ ]:
def sanity_overfit(arch: str, n_img: int = 8, steps: int = 80, thr: float = 0.85) -> float:
    seed_everything(0)
    S, recs = CFG["size"], get_frame(CFG["frame"])
    sub = recs["train"][:n_img]
    ds, items = SegSet(sub, S, aug="none"), EvalItems(sub, S)
    loader = torch.utils.data.DataLoader(ds, batch_size=len(sub), shuffle=True, num_workers=0)
    model, _ = build_model(arch, pretrained=True)
    model.to(DEVICE).train()
    crit = SegLoss("ce_dice", class_weights_from(sub, CFG["class_weight_cap"])).to(DEVICE)
    opt = torch.optim.AdamW([{"params": model.encoder_params(), "lr": 3e-4}, {"params": model.other_params(), "lr": 2e-3}], weight_decay=0.0)
    mb = probe_micro_batch(model, S, len(sub))
    it = 0
    while it < steps:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            for xc, yc in zip(xb.split(mb), yb.split(mb)):
                with AUTOCAST():
                    lg = model(xc)
                (crit(lg.float(), yc) * len(xc) / len(xb)).backward()
            opt.step(); it += 1
            if it >= steps:
                break
    res, _ = evaluate_models([model], items, "none")
    d = res.metrics()["macro_fg_dice"]
    verdict = "OK" if d >= thr else ("AVISO (modo smoke: poucos passos)" if MODE == "smoke" else "FALHA")
    print(f"Sanity overfit [{arch}] {n_img} imagens, {steps} passos: macro-fg Dice (treino) = {d:.3f} (limiar {thr}) -> {verdict}")
    del model, opt
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    if MODE != "smoke" and d < thr:
        raise RuntimeError(f"O modelo {arch} não consegue memorizar {n_img} imagens (Dice {d:.3f}<{thr}). Há um problema de pipeline: não prossiga.")
    return d


def lr_range_test(arch: str, steps: int = 60, lo: float = 1e-6, hi: float = 1e-1) -> float:
    seed_everything(0)
    S, recs = CFG["size"], get_frame(CFG["frame"])
    ds = SegSet(recs["train"], S, aug="standard")
    loader = torch.utils.data.DataLoader(ds, batch_size=min(8, len(ds)), shuffle=True, num_workers=0, drop_last=True)
    model, _ = build_model(arch, pretrained=True)
    model.to(DEVICE).train()
    crit = SegLoss("ce_dice", class_weights_from(recs["train"], CFG["class_weight_cap"])).to(DEVICE)
    opt = torch.optim.AdamW([{"params": model.encoder_params(), "lr": lo * 0.2}, {"params": model.other_params(), "lr": lo}], weight_decay=0.0)
    mult, lrs, losses, sm, it = (hi / lo) ** (1 / steps), [], [], None, 0
    while it < steps:
        for xb, yb in loader:
            lr = lo * mult ** it
            opt.param_groups[0]["lr"], opt.param_groups[1]["lr"] = lr * 0.2, lr
            opt.zero_grad(set_to_none=True)
            with AUTOCAST():
                lg = model(xb.to(DEVICE))
            loss = crit(lg.float(), yb.to(DEVICE))
            loss.backward(); opt.step()
            sm = float(loss) if sm is None else 0.8 * sm + 0.2 * float(loss)
            lrs.append(lr); losses.append(sm); it += 1
            if it >= steps or not math.isfinite(sm) or sm > 4 * min(losses):
                steps = it
                break
    best_lr = lrs[int(np.argmin(losses))] / 10
    fig, ax = plt.subplots(figsize=(5.5, 3.3))
    ax.semilogx(lrs, losses); ax.axvline(best_lr, color="r", ls="--", label=f"mín/10 = {best_lr:.1e}"); ax.legend(); ax.set_xlabel("LR do decoder"); ax.set_ylabel("loss (suavizada)")
    ax.set_title(f"LR range test — {arch}"); fig.tight_layout(); savefig(fig, f"lr_range_{arch}.png"); plt.show()
    print(f"LR sugerido (consultivo) para o decoder: {best_lr:.1e} | configurado: {CFG['lr_decoder']:.1e}")
    del model, opt
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return best_lr


if CFG["do_sanity"]:
    SANITY = {a: sanity_overfit(a) for a in CFG["archs"][:1]}
if CFG["do_lr_range"]:
    LR_SUGGESTED = lr_range_test(CFG["archs"][0])

## 14. Treino supervisionado — Experiências A e B (e sondas)

**Regras de seleção (só validação).** Em modo `full`, antes do treino principal, duas sondas curtas com a arquitetura de referência e uma só semente:
(i) *frame*/resolução (`padded 256` = réplica do artigo vs `roi 256` vs `roi 384`) e (ii) função de perda. Para não sobreajustar o ruído da validação
(16 imagens), só se abandona a configuração por omissão se a alternativa ganhar por **mais de 0,005** de macro-Dice dos tecidos.
Depois treinam-se **A** (arquitetura de referência) e **B** (restantes arquiteturas) com várias sementes; o relatório mostra média ± desvio.
Cada execução é idempotente (reutiliza `result.json`), por isso a sessão pode ser interrompida e retomada.

In [ ]:
ALL_RESULTS: list[dict] = []
REF_ARCH = CFG["archs"][0]
PROBE_MARGIN = 0.005


def val_score(r: dict) -> float:
    v = r["val"]["macro_fg_dice"]
    return -1.0 if v is None or np.isnan(v) else v


def pick(results: list[dict], key_fn, default_key):
    """Escolhe a melhor configuração, mas só abandona a por omissão se ganhar por > PROBE_MARGIN."""
    by = {key_fn(r): r for r in results}
    best_key = max(by, key=lambda k: val_score(by[k]))
    if default_key in by and val_score(by[best_key]) - val_score(by[default_key]) <= PROBE_MARGIN:
        return default_key, by
    return best_key, by


if CFG["do_probes"]:
    # --- sonda 1: frame/resolução
    res = []
    for fr, sz in CFG["probe_frames"]:
        if not BUDGET.ok(600):
            break
        res.append(train_run(default_spec(exp="probe_frame", arch=REF_ARCH, frame=fr, size=sz, epochs=CFG["probe_epochs"], patience=CFG["probe_epochs"], keep_ckpt=False)))
    ALL_RESULTS += res
    key, by = pick(res, lambda r: (r["spec"]["frame"], r["spec"]["size"]), (CFG["frame"], CFG["size"]))
    print("Sonda frame/resolução (val macro-fg Dice):", {k: round(val_score(v), 4) for k, v in by.items()}, "->", key)
    CFG["frame"], CFG["size"] = key
    # --- sonda 2: perda
    res = []
    for ls in CFG["probe_losses"]:
        if not BUDGET.ok(600):
            break
        res.append(train_run(default_spec(exp="probe_loss", arch=REF_ARCH, loss=ls, epochs=CFG["probe_epochs"], patience=CFG["probe_epochs"], keep_ckpt=False)))
    ALL_RESULTS += res
    key, by = pick(res, lambda r: r["spec"]["loss"], CFG["loss"])
    print("Sonda de perda (val macro-fg Dice):", {k: round(val_score(v), 4) for k, v in by.items()}, "->", key)
    CFG["loss"] = key
else:
    print(f"Sondas desativadas: frame={CFG['frame']}, size={CFG['size']}, loss={CFG['loss']} (valores por omissão, NÃO validados neste notebook)")

In [ ]:
SUP: list[dict] = []
for seed in CFG["seeds"]:               # sementes no ciclo exterior: se o tempo acabar, todas as arquiteturas têm pelo menos a 1.ª semente
    for arch in CFG["archs"]:
        if not BUDGET.ok(900):
            print("Orçamento de tempo quase esgotado: a parar novas execuções supervisionadas.")
            break
        SUP.append(train_run(default_spec(exp="sup", arch=arch, seed=seed)))
ALL_RESULTS += SUP
assert SUP, "Nenhuma execução supervisionada concluída"


def runs_table(results: list[dict]) -> pd.DataFrame:
    rows = []
    for r in results:
        v, s = r["val"], r["spec"]
        rows.append(dict(exp=s["exp"], arch=s["arch"], seed=s["seed"], frame=s["frame"], size=s["size"], loss=s["loss"], pretrained=r["provenance"]["pretrained_effective"],
                         best_ep=r["best_epoch"], epochs_run=r["epochs_run"], status=r["status"], macro_fg_dice=v["macro_fg_dice"], micro_fg_dice=v["micro_fg_dice"],
                         paper_dice=v["paper_dice"], dice_fibrin=v["dice_fibrin"], dice_granulation=v["dice_granulation"], dice_callus=v["dice_callus"],
                         macro_fg_iou=v["macro_fg_iou"], minutes=round(r["train_seconds"] / 60, 1)))
    return pd.DataFrame(rows)


SUP_TABLE = runs_table(SUP)
SUP_TABLE.to_csv(DIRS["tables"] / "supervised_runs_val.csv", index=False)
print("Validação (NÃO é o teste) — execuções supervisionadas:")
display(SUP_TABLE.round(4))
agg = SUP_TABLE.groupby("arch")[["macro_fg_dice", "micro_fg_dice", "paper_dice", "dice_fibrin", "dice_granulation", "dice_callus"]].agg(["mean", "std"]).round(4)
display(agg)
agg.to_csv(DIRS["tables"] / "supervised_by_arch_val.csv")
if not SUP_TABLE.pretrained.all():
    print("AVISO: há execuções SEM pesos pré-treinados (ver weights_provenance.json). Não são comparáveis a transfer learning.")

In [ ]:
# Curvas de aprendizagem (loss treino/val, Dice, IoU) para a melhor execução de cada arquitetura
best_by_arch = {}
for r in SUP:
    if r["spec"]["arch"] not in best_by_arch or val_score(r) > val_score(best_by_arch[r["spec"]["arch"]]):
        best_by_arch[r["spec"]["arch"]] = r
fig, axes = plt.subplots(len(best_by_arch), 3, figsize=(16, 3.4 * len(best_by_arch)), squeeze=False)
for row, (arch, r) in zip(axes, best_by_arch.items()):
    h = pd.DataFrame(r["history"])
    row[0].plot(h.epoch, h.train_loss, label="treino"); row[0].plot(h.epoch, h.val_loss, label="validação (EMA)"); row[0].set_title(f"{arch} — loss"); row[0].legend()
    for c in ["macro_fg_dice", "dice_fibrin", "dice_granulation", "dice_callus"]:
        row[1].plot(h.epoch, h[f"val_{c}"], label=c.replace("val_", ""))
    row[1].set_title("Dice de validação"); row[1].legend(fontsize=7); row[1].set_ylim(0, 1)
    row[2].plot(h.epoch, h.val_macro_fg_iou); row[2].set_title("macro-IoU (tecidos) de validação"); row[2].set_ylim(0, 1)
    for a in row:
        a.axvline(r["best_epoch"], color="k", ls=":", lw=0.8); a.set_xlabel("época")
fig.tight_layout(); savefig(fig, "learning_curves_supervised.png"); plt.show()

## 14b. Ablação (Experiência E) — indicativa
Retira-se **um** componente de cada vez à configuração de referência (mesma arquitetura, **uma só semente**, mesmo nº de épocas curto).
Com 16 imagens de validação e uma semente, diferenças pequenas são ruído: leia como indicação de ordem de grandeza, não como prova.

In [ ]:
ABL = []
if CFG["do_ablation"] and BUDGET.ok(1800):
    variants = {"referência": {}, "sem pré-treino": dict(pretrained=False), "sem augmentation (só flips)": dict(aug="flip"),
                "sem class-aware sampling": dict(sampler=False), "só CE (sem Dice)": dict(loss="ce"), "sem freeze inicial": dict(freeze_epochs=0)}
    for name, ov in variants.items():
        if not BUDGET.ok(600):
            break
        r = train_run(default_spec(exp="ablation:" + name, arch=REF_ARCH, epochs=CFG["ablation_epochs"], patience=CFG["ablation_epochs"], keep_ckpt=False, **ov))
        ABL.append(r)
    ALL_RESULTS += ABL
    abl_df = runs_table(ABL).assign(variante=[r["spec"]["exp"].split(":", 1)[1] for r in ABL])
    abl_df = abl_df[["variante", "macro_fg_dice", "dice_fibrin", "dice_granulation", "dice_callus", "micro_fg_dice", "paper_dice"]]
    abl_df["delta_macro_fg_vs_ref"] = abl_df.macro_fg_dice - abl_df.macro_fg_dice.iloc[0]
    abl_df.to_csv(DIRS["tables"] / "ablation_val.csv", index=False)
    display(abl_df.round(4))
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.barh(abl_df.variante[::-1], abl_df.macro_fg_dice[::-1]); ax.set_xlabel("macro-Dice dos tecidos (validação, 1 semente)")
    fig.tight_layout(); savefig(fig, "ablation_val.png"); plt.show()
else:
    print("Ablação não executada (`do_ablation=False` ou sem orçamento de tempo). A SSL e o TTA são avaliados abaixo contra a sua própria referência.")

## 16. Aprendizagem semissupervisionada (Experiência C)

Em vez da escolha aleatória de *pseudo-labels* do artigo, usa-se um **EMA *teacher–student* online** (Mean Teacher + FixMatch para segmentação),
partindo do **melhor checkpoint supervisionado** (escolhido na validação):

1. o *teacher* (EMA do *student*) prevê, com TTA de flips, a vista **fraca** de cada imagem **sem anotação** → probabilidade e rótulo por píxel;
2. **limiares por classe adaptativos**: para cada classe mantém-se a fração `ssl_keep_frac` de píxeis mais confiantes (percentil com média móvel),
   limitada a `[ssl_tau_min, ssl_tau_max]`; píxeis abaixo do limiar são **ignorados** (`ignore_index`) — evita que a fibrina (minoritária) seja
   engolida e reduz o viés de confirmação;
3. o *student* recebe a vista **forte** (fotometria forte + **CutMix** entre imagens, misturando também *pseudo-labels* e máscaras de confiança) e aprende com
   a perda de consistência, ponderada por um *ramp-up* sigmoidal, **somada** à perda supervisionada nas imagens anotadas (âncora anti-deriva);
4. avalia-se o *teacher* na validação a cada "época"; guarda-se o melhor (inclui o ponto inicial: se o SSL não melhorar, **mantém-se o supervisionado**);
5. **rondas iterativas**: o melhor da ronda inicializa a seguinte; se uma ronda não melhorar a validação, o processo pára e faz *rollback*.

Anti-fuga: os não anotados quase-duplicados de val/teste (dHash) foram removidos na auditoria; **nenhuma imagem/máscara de teste é lida aqui**.
Os *pseudo-labels* não têm GT, por isso a sua qualidade só é monitorizada por confiança e fração retida por classe (gráficos abaixo).

In [ ]:
class ClassThresholds:
    """Limiar de confiança por classe: percentil (1-keep_frac) das confianças dos píxeis previstos como essa classe, com média móvel e limites."""

    def __init__(self, tau_min: float, tau_max: float, keep_frac: float, momentum: float = 0.9):
        self.tau = np.full(K, (tau_min + tau_max) / 2)
        self.lo, self.hi, self.keep, self.m = tau_min, tau_max, keep_frac, momentum

    def update(self, conf: torch.Tensor, pl: torch.Tensor) -> None:
        for c in range(K):
            sel = conf[pl == c]
            if sel.numel() >= 200:
                q = float(torch.quantile(sel.float()[:2_000_000], 1 - self.keep))
                self.tau[c] = self.m * self.tau[c] + (1 - self.m) * float(np.clip(q, self.lo, self.hi))

    def as_tensor(self, device) -> torch.Tensor:
        return torch.tensor(self.tau, dtype=torch.float32, device=device)


def strong_photometric(x: torch.Tensor) -> torch.Tensor:
    """Fotometria forte em GPU (x em [0,1], (B,3,H,W)); o padding preto mantém-se preto."""
    B, dev = x.shape[0], x.device
    valid = (x.amax(1, keepdim=True) > 0).float()
    r = lambda lo, hi: torch.empty(B, 1, 1, 1, device=dev).uniform_(lo, hi)  # noqa: E731
    m = x.mean((1, 2, 3), keepdim=True)
    x = (x - m) * r(0.7, 1.3) + m + r(-0.15, 0.15)
    g = x.mean(1, keepdim=True)
    x = g + (x - g) * r(0.7, 1.3)
    x = x.clamp(1e-4, 1) ** r(0.7, 1.4)
    k = torch.tensor([1, 2, 1], device=dev, dtype=x.dtype)
    k = (k[:, None] * k[None]) / 16.0
    blur = F.conv2d(F.pad(x, (1, 1, 1, 1), mode="reflect"), k.expand(3, 1, 3, 3), groups=3)
    x = torch.where(torch.rand(B, 1, 1, 1, device=dev) < 0.3, blur, x)
    x = x + torch.randn_like(x) * r(0.0, 0.04) * (torch.rand(B, 1, 1, 1, device=dev) < 0.5)
    return x.clamp(0, 1) * valid


def cutmix(x: torch.Tensor, pl: torch.Tensor, keep: torch.Tensor):
    """CutMix: cola uma caixa de outra imagem do batch; mistura imagem, pseudo-label e máscara de confiança de forma consistente."""
    B, _, H, W = x.shape
    perm = torch.randperm(B, device=x.device)
    M = torch.zeros(B, 1, H, W, dtype=torch.bool, device=x.device)
    for i in range(B):
        area = random.uniform(0.15, 0.4) * H * W
        ar = random.uniform(0.5, 2.0)
        h, w = int(min(H, math.sqrt(area * ar))), int(min(W, math.sqrt(area / ar)))
        y0, x0 = random.randint(0, H - h), random.randint(0, W - w)
        M[i, :, y0:y0 + h, x0:x0 + w] = True
    return torch.where(M, x[perm], x), torch.where(M[:, 0], pl[perm], pl), torch.where(M[:, 0], keep[perm], keep)


def rampup(t: float) -> float:
    t = min(max(t, 0.0), 1.0)
    return math.exp(-5.0 * (1.0 - t) ** 2)


def _cycle(loader):
    while True:
        for b in loader:
            yield b


def train_ssl_round(sup_run: dict, init_ckpt: str, round_idx: int, unl_recs: list[dict]) -> dict:
    """Uma ronda de EMA teacher–student a partir de `init_ckpt`. Devolve resultado com o melhor teacher (inclui o ponto inicial)."""
    spec = dict(sup_run["spec"]) | dict(exp=f"ssl_r{round_idx}", epochs=CFG["ssl_epochs"], init=init_ckpt, keep_frac=CFG["ssl_keep_frac"], tau=[CFG["ssl_tau_min"], CFG["ssl_tau_max"]],
                lambda_u=CFG["ssl_lambda_u"], ssl_steps=CFG["ssl_steps_per_epoch"], n_unl=len(unl_recs))
    rid = run_id_of(spec)
    rdir = DIRS["runs"] / rid
    rf = rdir / "result.json"
    if rf.exists() and CFG["resume"]:
        print(f"[reutilizado] {rid}")
        return json.loads(rf.read_text())
    rdir.mkdir(parents=True, exist_ok=True)
    seed_everything(spec["seed"] + 1000 * (round_idx + 1))
    S, bs = spec["size"], spec["batch_size"]
    recs = get_frame(spec["frame"])
    lab_set, unl_set = SegSet(recs["train"], S, aug="standard"), SegSet(unl_recs, S, aug="weak", labeled=False)
    val_items = EvalItems(recs["val"], S)
    nw = CFG["num_workers"]
    mk = lambda ds, sampler: torch.utils.data.DataLoader(ds, batch_size=bs, sampler=sampler, num_workers=nw, drop_last=True, worker_init_fn=worker_init,  # noqa: E731
                                                          pin_memory=DEVICE.type == "cuda", persistent_workers=nw > 0)
    big = 10**6
    lab_w = torch.as_tensor(lab_set.sample_weights(CFG["fibrin_boost"]), dtype=torch.double)
    lab_it = _cycle(mk(lab_set, torch.utils.data.WeightedRandomSampler(lab_w, big, replacement=True)))
    unl_it = _cycle(mk(unl_set, torch.utils.data.RandomSampler(unl_set, replacement=True, num_samples=big)))

    student, prov = Net(ARCH_SPECS[spec["arch"]]["ctor"](None)), sup_run["provenance"]   # pesos vêm do checkpoint supervisionado
    student.load_state_dict(_load_half(Path(init_ckpt)))
    student.to(DEVICE)
    teacher = EMA(student, 0.99)
    micro = max(1, probe_micro_batch(student, S, bs) // 2)
    crit = SegLoss(spec["loss"], class_weights_from(recs["train"], CFG["class_weight_cap"]), CFG["label_smoothing"]).to(DEVICE)
    lr_mult = 0.25
    opt = torch.optim.AdamW([{"params": student.encoder_params(), "lr": spec["lr_encoder"] * lr_mult}, {"params": student.other_params(), "lr": spec["lr_decoder"] * lr_mult}],
                            weight_decay=CFG["weight_decay"])
    total = spec["epochs"] * spec["ssl_steps"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda st: lr_factor(st, int(0.05 * total), total, 0.05))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    thr = ClassThresholds(CFG["ssl_tau_min"], CFG["ssl_tau_max"], CFG["ssl_keep_frac"])

    base = evaluate_models([teacher.module], val_items, "none")[0].metrics()
    base_score = base["macro_fg_dice"]
    best, best_ep, bad = base_score, -1, 0
    shutil.copy(init_ckpt, rdir / "best.pt")
    hist, t0 = [], time.time()
    acc = dict(lsup=0.0, lunl=0.0, n=0, kept=np.zeros(K), seen=np.zeros(K), valid_frac=0.0)
    step = 0
    for epoch in range(spec["epochs"]):
        if not BUDGET.ok(60):
            break
        student.train()
        for _ in range(spec["ssl_steps"]):
            xl, yl = next(lab_it); xu = next(unl_it)
            xl, yl, xu = xl.to(DEVICE), yl.to(DEVICE), xu.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            for xc, yc in zip(xl.split(micro), yl.split(micro)):          # perda supervisionada (âncora)
                with AUTOCAST():
                    lg = student(xc)
                ls = crit(lg.float(), yc) * len(xc) / len(xl)
                scaler.scale(ls).backward(); acc["lsup"] += float(ls)
            with torch.no_grad():                                          # pseudo-labels do teacher (vista fraca, TTA de flips)
                p = torch.cat([canvas_probs(teacher.module, c, "flip") for c in xu.split(micro)])
                conf, pl = p.max(1)
            thr.update(conf, pl)
            keep = conf >= thr.as_tensor(DEVICE)[pl]
            for c in range(K):
                acc["seen"][c] += int((pl == c).sum()); acc["kept"][c] += int(((pl == c) & keep).sum())
            xs = strong_photometric(xu)
            xs, pl2, keep2 = cutmix(xs, pl, keep)
            tgt = torch.where(keep2, pl2, torch.full_like(pl2, IGNORE))
            w_u = spec["lambda_u"] * rampup(step / max(1.0, 0.3 * total))
            for xc, tc in zip(xs.split(micro), tgt.split(micro)):
                nvalid = int((tc != IGNORE).sum())
                if nvalid < 32:
                    continue
                with AUTOCAST():
                    lg = student(xc)
                lu = F.cross_entropy(lg.float(), tc, ignore_index=IGNORE) * (len(xc) / len(xs)) * w_u
                scaler.scale(lu).backward(); acc["lunl"] += float(lu)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(student.parameters(), CFG["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step(); teacher.update(student)
            acc["n"] += 1; step += 1
        vm = evaluate_models([teacher.module], val_items, "none")[0].metrics()
        score = 0.0 if np.isnan(vm["macro_fg_dice"]) else vm["macro_fg_dice"]
        hist.append(dict(epoch=epoch, loss_sup=acc["lsup"] / acc["n"], loss_unl=acc["lunl"] / acc["n"], val_macro_fg_dice=vm["macro_fg_dice"], val_micro_fg_dice=vm["micro_fg_dice"],
                         val_dice_fibrin=vm["dice_fibrin"], val_dice_granulation=vm["dice_granulation"], val_dice_callus=vm["dice_callus"], val_macro_fg_iou=vm["macro_fg_iou"],
                         val_paper_dice=vm["paper_dice"], w_u=w_u, **{f"tau_{CLASS_NAMES[c].lower()}": float(thr.tau[c]) for c in range(K)},
                         **{f"kept_{CLASS_NAMES[c].lower()}": float(acc["kept"][c] / max(acc["seen"][c], 1)) for c in range(K)},
                         **{f"share_{CLASS_NAMES[c].lower()}": float(acc["seen"][c] / max(acc["seen"].sum(), 1)) for c in range(K)}))
        acc = dict(lsup=0.0, lunl=0.0, n=0, kept=np.zeros(K), seen=np.zeros(K), valid_frac=0.0)
        if score > best + 1e-4:
            best, best_ep, bad = score, epoch, 0
            _save_half(teacher.module.state_dict(), rdir / "best.pt")
        else:
            bad += 1
        if epoch % max(1, spec["epochs"] // 8) == 0 or epoch == spec["epochs"] - 1:
            print(f"  {rid} ep{epoch:03d} Lsup {hist[-1]['loss_sup']:.3f} Lunl {hist[-1]['loss_unl']:.3f} val {score:.4f} (base {base_score:.4f}, best {best:.4f}@{best_ep})")
        if bad >= max(8, spec["epochs"] // 3):
            break

    final = Net(ARCH_SPECS[spec["arch"]]["ctor"](None)).to(DEVICE)
    final.load_state_dict(_load_half(rdir / "best.pt"))
    vfin = evaluate_models([final], val_items, "none")[0].metrics()
    result = dict(spec=spec, run_id=rid, status="finished", best_epoch=best_ep, epochs_run=len(hist), val=vfin, val_before=base, improved=best > base_score + 1e-4, history=hist,
                  provenance=prov, params_M=round(count_params(student), 2), micro_batch=micro, train_seconds=round(time.time() - t0, 1), ckpt=str(rdir / "best.pt"),
                  device=HW["gpu_name"] or "cpu")
    rf.write_text(json.dumps(result, indent=1, default=float))
    del student, teacher, opt, final
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result


SSL_RESULTS: list[dict] = []
SSL_FINAL: dict[str, dict] = {}   # id da execução supervisionada -> melhor resultado SSL (ou o supervisionado se não houve ganho)
if CFG["do_ssl"] and len(UNL) > 0 and BUDGET.ok(1800):
    arch_mean = SUP_TABLE.groupby("arch").macro_fg_dice.mean()
    ssl_arch = arch_mean.idxmax()
    cand = sorted([r for r in SUP if r["spec"]["arch"] == ssl_arch], key=val_score, reverse=True)
    targets = cand if CFG["ssl_all_seeds"] else cand[:1]
    unl_recs = load_unlabeled_records(CFG["frame"])
    print(f"SSL sobre {ssl_arch} | {len(unl_recs)} imagens sem anotação (após filtro anti-fuga) | alvos: {[t['run_id'] for t in targets]}")
    for tr in targets:
        cur, prev_score = tr, val_score(tr)
        for rnd in range(CFG["ssl_rounds"]):
            if not BUDGET.ok(1200):
                print("Orçamento de tempo: a parar rondas SSL."); break
            r = train_ssl_round(tr, cur["ckpt"], rnd, unl_recs)
            SSL_RESULTS.append(r)
            gain = val_score(r) - prev_score
            print(f"  ronda {rnd}: val macro-fg Dice {prev_score:.4f} -> {val_score(r):.4f} (Δ={gain:+.4f}) {'MELHOROU' if r['improved'] else 'sem ganho -> rollback e paragem'}")
            if not r["improved"]:
                break
            cur, prev_score = r, val_score(r)
        SSL_FINAL[tr["run_id"]] = cur
    ALL_RESULTS += SSL_RESULTS
elif not CFG["do_ssl"]:
    print("SSL desativado na configuração.")
else:
    print("SSL OMITIDO: sem imagens não anotadas utilizáveis (ou sem orçamento de tempo). O notebook continua em modo supervisionado.")

In [ ]:
if SSL_RESULTS:
    rows = [dict(run=r["run_id"], ronda=r["spec"]["exp"], antes=r["val_before"]["macro_fg_dice"], depois=r["val"]["macro_fg_dice"],
                 fibrin_antes=r["val_before"]["dice_fibrin"], fibrin_depois=r["val"]["dice_fibrin"], callus_antes=r["val_before"]["dice_callus"], callus_depois=r["val"]["dice_callus"],
                 granulation_antes=r["val_before"]["dice_granulation"], granulation_depois=r["val"]["dice_granulation"], melhorou=r["improved"], melhor_epoca=r["best_epoch"]) for r in SSL_RESULTS]
    SSL_TABLE = pd.DataFrame(rows)
    SSL_TABLE.to_csv(DIRS["tables"] / "ssl_before_after_val.csv", index=False)
    print("Antes vs depois do SSL (VALIDAÇÃO):")
    display(SSL_TABLE.round(4))
    r = SSL_RESULTS[0]
    h = pd.DataFrame(r["history"])
    fig, ax = plt.subplots(1, 4, figsize=(18, 3.4))
    ax[0].plot(h.epoch, h.loss_sup, label="supervisionada"); ax[0].plot(h.epoch, h.loss_unl, label="não supervisionada (ponderada)"); ax[0].legend(); ax[0].set_title("Perdas SSL (ronda 0)")
    for c in ["macro_fg_dice", "dice_fibrin", "dice_granulation", "dice_callus"]:
        ax[1].plot(h.epoch, h[f"val_{c}"], label=c)
    ax[1].axhline(r["val_before"]["macro_fg_dice"], color="k", ls=":", label="macro antes do SSL"); ax[1].legend(fontsize=7); ax[1].set_title("Dice de validação (teacher)")
    for c in CLASS_NAMES:
        ax[2].plot(h.epoch, h[f"kept_{c.lower()}"], label=c)
    ax[2].set_title("Fração de píxeis retidos por classe"); ax[2].legend(fontsize=7)
    for c in CLASS_NAMES:
        ax[3].plot(h.epoch, h[f"share_{c.lower()}"], label=c)
    ax[3].set_title("Proporção prevista por classe nos não anotados\n(deriva = sinal de viés de confirmação)"); ax[3].set_yscale("log"); ax[3].legend(fontsize=7)
    fig.tight_layout(); savefig(fig, "ssl_diagnostics.png"); plt.show()

## 17–18. Comparação de experiências, *ensemble*, TTA e pós-processamento (Experiência D) — **só validação**

Constrói-se um *pool* com os melhores checkpoints (supervisionados e SSL) e enumeram-se **candidatos**: melhor modelo único supervisionado,
*ensembles* por arquitetura (sementes), melhor SSL, *ensemble* supervisionado completo e *ensemble* total. Para cada candidato escolhem-se na
**validação** o TTA (`none`/`flip`/`flip_ms`) e a área mínima de componentes (com a mesma histerese de 0,002 para não ajustar ruído).
As probabilidades são **médias uniformes** (afinar pesos numa validação de 16 imagens sobreajustaria). O candidato final é o de maior
macro-Dice dos tecidos na validação; empates resolvem-se pelo menor nº de modelos.

**O teste continua intocado nesta célula.** As probabilidades de validação são calculadas uma vez por (modelo, TTA) e reutilizadas.

In [ ]:
POOL: dict[str, dict] = {}
for r in SUP:
    POOL[r["run_id"]] = dict(run=r, kind="sup", arch=r["spec"]["arch"], seed=r["spec"]["seed"], ckpt=r["ckpt"])
for sup_id, r in SSL_FINAL.items():
    if r["run_id"] != sup_id:                               # só entra se houve de facto SSL com ganho
        POOL[r["run_id"]] = dict(run=r, kind="ssl", arch=r["spec"]["arch"], seed=r["spec"]["seed"], ckpt=r["ckpt"])
SSL_IDS = [k for k, v in POOL.items() if v["kind"] == "ssl"]
SUP_IDS = [k for k, v in POOL.items() if v["kind"] == "sup"]
print(f"Pool: {len(SUP_IDS)} supervisionados, {len(SSL_IDS)} SSL (com ganho na validação)")

SIZE, FRAME = CFG["size"], CFG["frame"]
_recs = get_frame(FRAME)
VAL_ITEMS = EvalItems(_recs["val"], SIZE)
_MODEL_CACHE: dict[str, Net] = {}


def load_pool_model(rid: str) -> Net:
    if rid not in _MODEL_CACHE:
        m = Net(ARCH_SPECS[POOL[rid]["arch"]]["ctor"](None)).to(DEVICE).eval()
        m.load_state_dict(_load_half(Path(POOL[rid]["ckpt"])))
        _MODEL_CACHE[rid] = m
    return _MODEL_CACHE[rid]


_PROBS_VAL: dict[tuple, list] = {}


def val_probs(rid: str, tta: str) -> list[np.ndarray]:
    if (rid, tta) not in _PROBS_VAL:
        _PROBS_VAL[(rid, tta)] = predict_probs([load_pool_model(rid)], VAL_ITEMS, tta)
    return _PROBS_VAL[(rid, tta)]


def ens_probs(ids: list[str], tta: str, getter) -> list[np.ndarray]:
    per = [getter(i, tta) for i in ids]
    return [np.mean([p[j] for p in per], axis=0) for j in range(len(per[0]))]


def candidates() -> dict[str, list[str]]:
    c: dict[str, list[str]] = {}
    best_sup = max(SUP_IDS, key=lambda i: val_score(POOL[i]["run"]))
    c["sup_best_single"] = [best_sup]
    for a in CFG["archs"]:
        ids = [i for i in SUP_IDS if POOL[i]["arch"] == a]
        if len(ids) > 1:
            c[f"sup_seeds_{a}"] = ids
    per_arch_best = [max((i for i in SUP_IDS if POOL[i]["arch"] == a), key=lambda i: val_score(POOL[i]["run"])) for a in CFG["archs"] if any(POOL[i]["arch"] == a for i in SUP_IDS)]
    if len(per_arch_best) > 1:
        c["sup_best_per_arch"] = per_arch_best
    if len(SUP_IDS) > 1:
        c["sup_all"] = SUP_IDS
    if SSL_IDS:
        c["ssl_best_single"] = [max(SSL_IDS, key=lambda i: val_score(POOL[i]["run"]))]
        c["all_models"] = SUP_IDS + SSL_IDS
        ssl_arch_ids = [i for i in SUP_IDS if POOL[i]["arch"] == POOL[SSL_IDS[0]]["arch"]]
        c["ssl+sup_same_arch"] = list(dict.fromkeys(SSL_IDS + ssl_arch_ids))
    return {k: list(dict.fromkeys(v)) for k, v in c.items()}


CANDS = candidates()
rows, TUNED = [], {}
for name, ids in CANDS.items():
    if not BUDGET.ok(300):
        print("Orçamento de tempo: candidatos restantes ignorados."); break
    best = (-1, None)
    per_tta = {}
    for tta in CFG["tta_candidates"]:
        pr = ens_probs(ids, tta, val_probs)
        sc = evaluate_preds(VAL_ITEMS.names, VAL_ITEMS.gts, probs_to_preds(pr)).metrics()["macro_fg_dice"]
        per_tta[tta] = sc
        if sc > best[0] + 1e-9 and (best[1] is None or sc - best[0] > 0.002):   # histerese: só troca se ganhar > 0,002
            best = (sc, tta)
    tta = best[1] or "none"
    pr = ens_probs(ids, tta, val_probs)
    base_pp = (evaluate_preds(VAL_ITEMS.names, VAL_ITEMS.gts, probs_to_preds(pr, 0.0)).metrics()["macro_fg_dice"], 0.0)
    pp_best = base_pp
    for fr in CFG["pp_min_area_fracs"][1:]:
        sc = evaluate_preds(VAL_ITEMS.names, VAL_ITEMS.gts, probs_to_preds(pr, fr)).metrics()["macro_fg_dice"]
        if sc > pp_best[0] + 0.002:
            pp_best = (sc, fr)
    res = evaluate_preds(VAL_ITEMS.names, VAL_ITEMS.gts, probs_to_preds(pr, pp_best[1])).metrics()
    TUNED[name] = dict(ids=ids, tta=tta, pp=pp_best[1], val=res)
    rows.append(dict(candidato=name, n_modelos=len(ids), tta=tta, pp_min_area=pp_best[1], **{f"val_macro_fg(TTA={k})": round(v, 4) for k, v in per_tta.items()},
                     val_macro_fg=res["macro_fg_dice"], val_micro_fg=res["micro_fg_dice"], val_paper_dice=res["paper_dice"], val_fibrin=res["dice_fibrin"],
                     val_granulation=res["dice_granulation"], val_callus=res["dice_callus"]))
CAND_TABLE = pd.DataFrame(rows).sort_values(["val_macro_fg", "n_modelos"], ascending=[False, True]).reset_index(drop=True)
CAND_TABLE.to_csv(DIRS["tables"] / "candidates_val.csv", index=False)
print("Candidatos (VALIDAÇÃO; a seleção usa só estas colunas — o teste não foi lido):")
display(CAND_TABLE.round(4))
FINAL_NAME = CAND_TABLE.candidato.iloc[0]
FINAL = TUNED[FINAL_NAME]
print(f"\nCandidato final escolhido pela validação: {FINAL_NAME} | modelos={FINAL['ids']} | TTA={FINAL['tta']} | pp_min_area={FINAL['pp']}")
print("Nota: com 16 imagens de validação e vários candidatos, o valor de validação do vencedor é otimista (viés de seleção).")

## 19. Avaliação final no **teste** (uma única vez)

Cada acesso ao teste fica registado em `TEST_LEDGER` (modelo, TTA, utilização). Avaliam-se, **sem qualquer ajuste posterior**:
(1) o melhor modelo único supervisionado (sem TTA nem pós-processamento) — comparável à fase SL do artigo;
(2) o melhor SSL (se existir), sem TTA — comparável à fase SSL;
(3) o candidato final escolhido na validação, com o seu TTA e pós-processamento. Os intervalos de confiança de 95 % são *bootstrap* sobre as imagens de teste.

In [ ]:
TEST_ITEMS = EvalItems(_recs["test"], SIZE)
TEST_LEDGER: list[dict] = []
_PROBS_TEST: dict[tuple, list] = {}


def test_probs(rid: str, tta: str) -> list[np.ndarray]:
    if (rid, tta) not in _PROBS_TEST:
        TEST_LEDGER.append(dict(event="forward_on_test", model=rid, tta=tta, t=time.strftime("%H:%M:%S")))
        _PROBS_TEST[(rid, tta)] = predict_probs([load_pool_model(rid)], TEST_ITEMS, tta)
    return _PROBS_TEST[(rid, tta)]


def evaluate_test(label: str, ids: list[str], tta: str, pp: float) -> EvalResult:
    TEST_LEDGER.append(dict(event="evaluate", label=label, models=ids, tta=tta, pp=pp, t=time.strftime("%H:%M:%S")))
    pr = ens_probs(ids, tta, test_probs)
    return evaluate_preds(TEST_ITEMS.names, TEST_ITEMS.gts, probs_to_preds(pr, pp)), pr


TEST_ROWS: dict[str, dict] = {}
best_sup_id = CANDS["sup_best_single"][0]
spec_rows = [("Nosso: melhor supervisionado (1 modelo, sem TTA)", [best_sup_id], "none", 0.0)]
if "ssl_best_single" in CANDS:
    spec_rows.append(("Nosso: melhor SSL (1 modelo, sem TTA)", CANDS["ssl_best_single"], "none", 0.0))
spec_rows.append((f"Nosso: FINAL = {FINAL_NAME} (TTA={FINAL['tta']}, pp={FINAL['pp']})", FINAL["ids"], FINAL["tta"], FINAL["pp"]))
for label, ids, tta, pp in spec_rows:
    res, pr = evaluate_test(label, ids, tta, pp)
    TEST_ROWS[label] = dict(res=res, probs=pr, ids=ids, tta=tta, pp=pp, metrics=res.metrics())
FINAL_LABEL = spec_rows[-1][0]
print(f"Leituras do teste registadas: {len(TEST_LEDGER)} eventos")
display(pd.DataFrame(TEST_LEDGER).drop(columns=["t"], errors="ignore"))

## 21. Tabela comparativa com o artigo
* **Global (comparável ao artigo)** = `paper_*`: réplica do protocolo do código dos autores (ver secção 12). Mesmo assim, equivalência **não garantida**: dados de teste
  iguais (16 imagens, divisão oficial), mas frame/protocolo exato do artigo, sementes e checkpoints são desconhecidos, e o teste é minúsculo.
* `micro_fg` e `macro_fg` são as definições padrão, mais exigentes, e **não** são comparáveis ao 87,64 do artigo.
* Por classe: `paper_dice_<c>` usa a convenção por imagem do código dos autores; `dice_<c>` é o Dice agregado sobre o dataset (padrão).

In [ ]:
KEYS_CI = ["paper_dice", "micro_fg_dice", "macro_fg_dice", "paper_dice_fibrin", "paper_dice_granulation", "paper_dice_callus", "dice_fibrin", "dice_granulation", "dice_callus"]
for label, row in TEST_ROWS.items():
    row["ci"] = row["res"].bootstrap(KEYS_CI, CFG["bootstrap_b"], seed=0)


def fmt(m: dict, ci: dict | None, key: str) -> str:
    v = m[key]
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "n/a"
    s = f"{100 * v:.2f}"
    if ci and key in ci:
        s += f" [{100 * ci[key][0]:.1f}–{100 * ci[key][1]:.1f}]"
    return s


cmp_rows = [
    dict(modelo="ARTIGO (reportado) — SL", dice_global_paper="84.89", IoU_paper="73.75", Precision_paper="87.69", Recall_paper="82.27", micro_fg="—", macro_fg="—", fibrin_paperconv="—", granulation_paperconv="—", callus_paperconv="—"),
    dict(modelo="ARTIGO (reportado) — SSL", dice_global_paper="87.64", IoU_paper="77.99", Precision_paper="88.14", Recall_paper="87.14", micro_fg="—", macro_fg="—",
         fibrin_paperconv="69.01", granulation_paperconv="94.11", callus_paperconv="78.27"),
]
for label, row in TEST_ROWS.items():
    m, ci = row["metrics"], row["ci"]
    cmp_rows.append(dict(modelo=label, dice_global_paper=fmt(m, ci, "paper_dice"), IoU_paper=fmt(m, None, "paper_iou"), Precision_paper=fmt(m, None, "paper_precision"),
                         Recall_paper=fmt(m, None, "paper_recall"), micro_fg=fmt(m, ci, "micro_fg_dice"), macro_fg=fmt(m, ci, "macro_fg_dice"),
                         fibrin_paperconv=fmt(m, ci, "paper_dice_fibrin"), granulation_paperconv=fmt(m, ci, "paper_dice_granulation"), callus_paperconv=fmt(m, ci, "paper_dice_callus")))
COMPARISON = pd.DataFrame(cmp_rows)
COMPARISON.to_csv(DIRS["tables"] / "final_comparison_test.csv", index=False)
print("TESTE (16 imagens) — valores em %, intervalo de confiança bootstrap 95% entre parênteses retos:")
display(COMPARISON)

cls_rows = []
for label, row in TEST_ROWS.items():
    m, ci = row["metrics"], row["ci"]
    for c in ["fibrin", "granulation", "callus"]:
        cls_rows.append(dict(modelo=label, classe=c, dice_agregado=fmt(m, ci, f"dice_{c}"), dice_convencao_artigo=fmt(m, ci, f"paper_dice_{c}"),
                             iou_agregado=fmt(m, None, f"iou_{c}"), precision_agregada=fmt(m, None, f"precision_{c}"), recall_agregado=fmt(m, None, f"recall_{c}"),
                             dice_artigo_SSL=PAPER["per_class_ssl"][c.capitalize()]["Dice"]))
display(pd.DataFrame(cls_rows))

# Veredicto calculado (nunca pré-preenchido)
fm, fci = TEST_ROWS[FINAL_LABEL]["metrics"], TEST_ROWS[FINAL_LABEL]["ci"]
verdict_rows = []
for key, ref, nm in [("paper_dice", 0.8764, "Dice global (protocolo-réplica) vs SSL do artigo 87,64"), ("paper_dice_fibrin", 0.6901, "Dice Fibrin (conv. artigo) vs 69,01"),
                     ("paper_dice_granulation", 0.9411, "Dice Granulation (conv. artigo) vs 94,11"), ("paper_dice_callus", 0.7827, "Dice Callus (conv. artigo) vs 78,27")]:
    v, lo, hi = fm[key], *fci[key]
    verdict_rows.append(dict(comparação=nm, nosso=round(100 * v, 2), IC95=f"[{100 * lo:.1f}, {100 * hi:.1f}]", artigo=round(100 * ref, 2),
                             ponto_acima=bool(v > ref), IC_exclui_artigo=bool(lo > ref or hi < ref)))
VERDICT = pd.DataFrame(verdict_rows)
display(VERDICT)
print("Leitura: 'ponto_acima' compara só a estimativa pontual; só se pode falar de melhoria sólida se o IC exclui o valor do artigo (e mesmo assim"
      " a comparação depende da equivalência do protocolo e de um teste de 16 imagens, possivelmente com várias imagens do mesmo doente).")

## 20. Visualização das previsões no teste
Imagem, GT, previsão e *overlay* transparente; melhores e piores casos segundo o Dice médio dos tecidos por imagem (GT∪predição).

In [ ]:
FINAL_ROW = TEST_ROWS[FINAL_LABEL]
FINAL_PREDS = probs_to_preds(FINAL_ROW["probs"], FINAL_ROW["pp"])


def per_image_table(res: EvalResult, items: EvalItems) -> pd.DataFrame:
    rows = []
    for i, cm in enumerate(res.cms):
        pc = per_class_from_cm(cm)
        d = pc["dice"][TISSUES]
        rows.append(dict(name=items.names[i], h=items.geoms[i]["h"], w=items.geoms[i]["w"], fg_frac=cm[1:].sum() / cm.sum(),
                         dice_fibrin=d[0], dice_granulation=d[1], dice_callus=d[2], mean_dice=np.nanmean(d) if (~np.isnan(d)).any() else np.nan,
                         gt_fibrin_px=int(cm[1].sum()), pred_fibrin_px=int(cm[:, 1].sum())))
    return pd.DataFrame(rows)


PER_IMG = per_image_table(FINAL_ROW["res"], TEST_ITEMS)
PER_IMG.to_csv(DIRS["tables"] / "test_per_image_final.csv", index=False)


def show_predictions(items: EvalItems, preds: list[np.ndarray], idxs: list[int], title: str, fname: str, ncols: int = 4) -> None:
    n = len(idxs)
    fig, axes = plt.subplots(n, ncols, figsize=(3.1 * ncols, 3.1 * n), squeeze=False)
    for r, i in enumerate(idxs):
        img, gt, pr = items.imgs[i], items.gts[i], preds[i]
        for a, im, t in zip(axes[r], [img, colorize(gt), colorize(pr), overlay(img, pr)], ["imagem", "GT", "previsão", "overlay (previsão)"]):
            a.imshow(im); a.axis("off"); a.set_title(f"{items.names[i]} — {t}" if t == "imagem" else t, fontsize=8)
        d = PER_IMG.iloc[i]
        axes[r, 2].set_title(f"previsão (Dice méd. {d.mean_dice:.2f})", fontsize=8)
    fig.suptitle(title); fig.tight_layout(); savefig(fig, fname); plt.show()


order = PER_IMG.mean_dice.fillna(-1).sort_values().index.tolist()
show_predictions(TEST_ITEMS, FINAL_PREDS, order[-4:][::-1], "Teste — melhores 4 casos (modelo final)", "test_best_cases.png")
show_predictions(TEST_ITEMS, FINAL_PREDS, order[:4], "Teste — piores 4 casos (modelo final)", "test_worst_cases.png")

In [ ]:
# Matriz de confusão por píxel (agregada no teste, linhas normalizadas)
cm_sum = FINAL_ROW["res"].cms.sum(0)
cm_norm = cm_sum / cm_sum.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for a, mat, ttl, f in [(ax[0], cm_norm, "normalizada por linha (recall)", ".2f"), (ax[1], np.log10(cm_sum + 1), "log10(contagem+1)", ".1f")]:
    im = a.imshow(mat, cmap="Blues"); a.set_xticks(range(K)); a.set_yticks(range(K)); a.set_xticklabels(CLASS_NAMES, rotation=30); a.set_yticklabels(CLASS_NAMES)
    a.set_xlabel("previsto"); a.set_ylabel("GT"); a.set_title(f"Matriz de confusão (teste) — {ttl}")
    for i in range(K):
        for j in range(K):
            a.text(j, i, format(mat[i, j], f), ha="center", va="center", color="white" if mat[i, j] > mat.max() * 0.6 else "black", fontsize=8)
fig.tight_layout(); savefig(fig, "test_confusion_matrix.png"); plt.show()

# Gráficos comparativos de modelos (teste) com IC
fig, ax = plt.subplots(figsize=(9, 3.6))
labels = list(TEST_ROWS)
for off, key in enumerate(["dice_fibrin", "dice_granulation", "dice_callus"]):
    vals = [TEST_ROWS[l]["metrics"][key] for l in labels]
    err = np.array([[TEST_ROWS[l]["metrics"][key] - TEST_ROWS[l]["ci"][key][0], TEST_ROWS[l]["ci"][key][1] - TEST_ROWS[l]["metrics"][key]] for l in labels]).T
    ax.bar(np.arange(len(labels)) + 0.27 * (off - 1), vals, 0.27, yerr=err, label=key.replace("dice_", ""), capsize=2)
ax.set_xticks(range(len(labels))); ax.set_xticklabels([l.split(":")[1].split("(")[0].strip()[:22] for l in labels], fontsize=7); ax.set_ylim(0, 1); ax.legend(); ax.set_title("Dice por classe no teste (agregado; IC bootstrap 95 %)")
fig.tight_layout(); savefig(fig, "test_model_comparison.png"); plt.show()

## 22. Análise de erros e limitações
O texto abaixo é **gerado a partir dos números desta execução** (nenhuma frase foi escrita de antemão sobre resultados).

In [ ]:
def boundary_error_share(gts, preds, band: int = 3) -> float:
    """Fração dos píxeis errados (excluindo erros só de fundo<->fundo) que estão a <= `band` px de uma fronteira entre classes no GT."""
    err_tot = err_near = 0
    for g, p in zip(gts, preds):
        edge = np.zeros(g.shape, np.uint8)
        for c in range(K):
            m = (g == c).astype(np.uint8)
            edge |= (cv2.dilate(m, np.ones((3, 3), np.uint8)) - cv2.erode(m, np.ones((3, 3), np.uint8)))
        near = cv2.dilate(edge, np.ones((2 * band + 1, 2 * band + 1), np.uint8)).astype(bool)
        err = g != p
        err_tot += int(err.sum()); err_near += int((err & near).sum())
    return err_near / max(err_tot, 1)


off = cm_sum.astype(float).copy(); np.fill_diagonal(off, 0)
conf_pairs = sorted(((off[i, j] / cm_sum[i].sum(), CLASS_NAMES[i], CLASS_NAMES[j]) for i in range(1, K) for j in range(K) if i != j), reverse=True)[:4]
bshare = boundary_error_share(TEST_ITEMS.gts, FINAL_PREDS)
size_corr = PER_IMG[["h", "mean_dice"]].dropna().corr(method="spearman").iloc[0, 1] if PER_IMG.mean_dice.notna().sum() > 2 else float("nan")
fib_img = PER_IMG[PER_IMG.gt_fibrin_px > 0]
print("Principais confusões (fração dos píxeis de GT da classe, linha->coluna):")
for v, a, b in conf_pairs:
    print(f"  {a:12s} -> {b:12s}: {100 * v:5.1f} %")
print(f"Fração dos píxeis errados a <=3 px de uma fronteira do GT: {100 * bshare:.1f} %  (alta => erros dominados por ambiguidade de fronteira)")
print(f"Correlação de Spearman entre tamanho da imagem original e Dice médio por imagem: {size_corr:.2f}")
print(f"Fibrina: {len(fib_img)}/{len(PER_IMG)} imagens de teste têm fibrina no GT; Dice mediano por imagem nessas: {fib_img.dice_fibrin.median():.2f}; "
      f"piores: {', '.join(fib_img.sort_values('dice_fibrin').name.head(3))}")
print(f"Fibrina prevista vs GT (píxeis, teste): {int(PER_IMG.pred_fibrin_px.sum())} vs {int(PER_IMG.gt_fibrin_px.sum())} "
      f"-> {'sobre-segmentação' if PER_IMG.pred_fibrin_px.sum() > 1.15 * PER_IMG.gt_fibrin_px.sum() else ('sub-segmentação' if PER_IMG.pred_fibrin_px.sum() < 0.85 * PER_IMG.gt_fibrin_px.sum() else 'volume comparável')}")
display(PER_IMG.round(3).sort_values("mean_dice").head(8))

**Limitações (válidas independentemente dos números):**
1. **Teste de 16 imagens**: os IC *bootstrap* são largos; diferenças de 1–3 pontos de Dice não são estatisticamente distinguíveis. Imagens do mesmo doente podem
   estar em divisões diferentes (não há identificador de doente) — a generalização pode estar sobrestimada, também no artigo.
2. **Equivalência de protocolo não garantida**: o `paper_*` replica o código dos autores, mas não temos confirmação de que as tabelas do artigo foram geradas exatamente
   assim, nem dos checkpoints/sementes deles. Em particular, o acumulador não reposto torna esse "global" diferente de um micro-Dice padrão.
3. **Resolução**: as imagens originais são pequenas (67–266 px); ampliar para 256/384 não recupera detalhe, apenas dá mais *tokens* úteis ao encoder.
4. **Seleção em 16 imagens de validação** (mitigada com histerese, sementes múltiplas e ensemble uniforme) pode ainda sobreajustar a validação.
5. **SSL**: não há GT para os não anotados; a qualidade dos *pseudo-labels* só é monitorizada indiretamente. O ganho tem de ser lido contra o ponto inicial supervisionado.
6. **Pesos pré-treinados**: se a execução registou `random_init` em `weights_provenance.json`, os resultados **não** representam *transfer learning*.

**Próxima experiência mais promissora** (depende dos resultados acima): validação cruzada estratificada *k*-fold sobre as 94 imagens de desenvolvimento (sem tocar no teste)
para selecionar hiperparâmetros com menos ruído; resolução 384 se a sonda a favoreceu; mais sementes no *ensemble*; ou *self-supervised pre-training* nos 600 não anotados se
o SSL se mostrar limitado pela qualidade das *features*.

## 23. Inferência sobre uma imagem nova
`segment_image` aceita um caminho ou um *array* RGB, faz *letterbox*, TTA, *ensemble* do candidato final, reverte para o tamanho original e aplica o pós-processamento
escolhido na validação. Para fotografias novas, `assume_zero_padded=False` (a imagem inteira é o conteúdo). As imagens do dataset estão coladas num canto preto, por
isso a demonstração usa `assume_zero_padded=True` (recorta a extensão do conteúdo).

In [ ]:
def segment_image(image, ids: list[str] | None = None, tta: str | None = None, pp: float | None = None, assume_zero_padded: bool = False, return_probs: bool = False):
    ids, tta, pp = ids or FINAL["ids"], tta or FINAL["tta"], FINAL["pp"] if pp is None else pp
    img = read_rgb(Path(image)) if isinstance(image, (str, Path)) else np.asarray(image)
    if img.ndim == 2:
        img = np.stack([img] * 3, -1)
    img = img[..., :3].astype(np.uint8)
    if assume_zero_padded:
        h, w = content_extent(img)
        img = np.ascontiguousarray(img[:h, :w])
    items = EvalItems([dict(name="new", img=img, mask=None)], SIZE, labeled=False)
    probs = predict_probs([load_pool_model(i) for i in ids], items, tta)[0]
    pred = remove_small_components(probs.argmax(0).astype(np.uint8), pp)
    return (img, pred, probs) if return_probs else (img, pred)


demo = []
if len(UNL):
    demo.append(("não anotada (dataset)", UNL[0]["path"], True))
demo.append(("teste (apenas visualização)", _locate("Padded", "Images", SPLIT_NAMES["test"][0]), True))
fig, axes = plt.subplots(len(demo), 3, figsize=(10, 3.4 * len(demo)), squeeze=False)
for r, (nm, path, zp) in enumerate(demo):
    img, pred = segment_image(path, assume_zero_padded=zp)
    for a, im, t in zip(axes[r], [img, colorize(pred), overlay(img, pred)], [f"{nm}", "segmentação", "overlay"]):
        a.imshow(im); a.axis("off"); a.set_title(t, fontsize=9)
fig.tight_layout(); savefig(fig, "inference_demo.png"); plt.show()
print("Fração de píxeis por classe na imagem demo:", dict(zip(CLASS_NAMES, np.round(np.bincount(pred.ravel(), minlength=K) / pred.size, 3))))

## 24. Exportação de resultados e modelos
Em `OUT`: `config.json` (configuração, sementes, versões, hardware), `weights_provenance.json`, `tables/*.csv|json`, `figures/*.png`, `runs/*/result.json` (histórico de treino),
`models/*.pt` (estados em fp16 do candidato final) e `final_bundle.json` (como reconstruir o *ensemble*). Em Kaggle ficam em `/kaggle/working`.

In [ ]:
(OUT / "config.json").write_text(json.dumps(dict(cfg=CFG, hw=HW, seeds=CFG["seeds"], mode=MODE, data_root=str(DATA_ROOT), split_sizes={k: len(v) for k, v in SPLIT_NAMES.items()},
                                                 n_unlabeled_used=len(UNL), n_unlabeled_excluded_dups=len(EXCLUDED_UNL)), indent=1, default=str))
(OUT / "weights_provenance.json").write_text(json.dumps(WEIGHTS_PROVENANCE, indent=1, default=str))
(OUT / "test_ledger.json").write_text(json.dumps(TEST_LEDGER, indent=1))
final_members = []
for rid in FINAL["ids"]:
    dst = DIRS["models"] / f"{rid}.pt"
    shutil.copy(POOL[rid]["ckpt"], dst)
    final_members.append(dict(file=f"models/{dst.name}", arch=POOL[rid]["arch"], kind=POOL[rid]["kind"], seed=POOL[rid]["seed"], val_macro_fg_dice=val_score(POOL[rid]["run"])))
(OUT / "final_bundle.json").write_text(json.dumps(dict(
    candidate=FINAL_NAME, members=final_members, frame=FRAME, size=SIZE, tta=FINAL["tta"], pp_min_area_frac=FINAL["pp"], classes=CLASS_NAMES, mean=IMAGENET_MEAN, std=IMAGENET_STD,
    note="Modelos = smp.<arch>(encoder_weights=None, classes=4) envolvidos em Net (normalização ImageNet interna); state_dict em fp16 -> converter para float32 ao carregar.",
    test_metrics={k: v for k, v in FINAL_ROW["metrics"].items()}, test_ci_95={k: list(v) for k, v in FINAL_ROW["ci"].items()}), indent=1, default=float))
pd.DataFrame([dict(label=l, **r["metrics"]) for l, r in TEST_ROWS.items()]).to_csv(DIRS["tables"] / "test_metrics_all_definitions.csv", index=False)
all_hist = [dict(run_id=r["run_id"], **h) for r in ALL_RESULTS for h in r["history"]]
pd.DataFrame(all_hist).to_csv(DIRS["tables"] / "training_history_all_runs.csv", index=False)
runs_table([r for r in ALL_RESULTS if "val" in r]).to_csv(DIRS["tables"] / "all_runs_val.csv", index=False)
(DIRS["tables"] / "paper_reported_constants.json").write_text(json.dumps(PAPER, indent=1))

def md_table(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except ImportError:  # `tabulate` ausente: texto simples
        return "```\n" + df.to_string(index=False) + "\n```"


report = [f"# Relatório de experiências — modo `{MODE}`", "", f"* Hardware: {HW['gpu_name'] or 'CPU'} ({HW['vram_gb']} GB) | torch {HW['torch']} | smp {HW['smp']} | timm {HW['timm']}",
          f"* Frame/resolução: `{FRAME}`/{SIZE} | perda: `{CFG['loss']}` | sementes: {CFG['seeds']} | arquiteturas: {CFG['archs']}",
          f"* Pesos pré-treinados efetivos: {{{', '.join(f'{k}: {v.get('pretrained_effective')}' for k, v in WEIGHTS_PROVENANCE.items())}}}",
          f"* Não anotadas usadas: {len(UNL)} (excluídas por quase-duplicado de val/teste: {len(EXCLUDED_UNL)})", f"* Candidato final (escolhido na validação): `{FINAL_NAME}` — {len(FINAL['ids'])} modelo(s), TTA={FINAL['tta']}, pp={FINAL['pp']}",
          f"* Eventos de leitura do teste: {len(TEST_LEDGER)} (ver test_ledger.json)", "", "## Tabela comparativa (teste; % ; IC bootstrap 95 %)", "", md_table(COMPARISON),
          "", "## Veredicto calculado", "", VERDICT.to_string(index=False), "",
          "Aviso: os valores acima só têm valor científico se `weights_provenance.json` indicar pesos pré-treinados e se o modo não for `smoke`."]
(OUT / "experiments_report.md").write_text("\n".join(report))
print("Exportado em", OUT)
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.suffix in (".json", ".csv", ".md", ".png") and "runs" not in p.parts:
        print("  ", p.relative_to(OUT), f"{p.stat().st_size / 1024:.0f} KB")
print("\nMODO:", MODE, "| " + ("ATENÇÃO: modo smoke = teste de pipeline; estes números NÃO têm valor científico." if MODE == "smoke" else "execução científica."))